# Daily Full Funnel Performance Query - All Volume

For all compass and net call steps, gets throughput/breakage/queue rates, and the volume that made it through.

Additionally, have expected conversion and financial plan information. 

Aggregating all data

In [0]:
!pip install tqdm
!pip install -U python-docx
!pip install openai
!pip install dotenv

# Creating KPI JSON

### Data Aggregation

In [0]:
from typing import Optional, Union, Sequence
from pyspark.sql import DataFrame
from datetime import date, timedelta

def _compute_yday_and_prior4_dates(today: Optional[date] = None):
    if today is None:
        today = date.today()
    yday = today - timedelta(days=1)
    dates = [yday - timedelta(days=7 * i) for i in range(0, 5)]
    return [d.strftime("%Y-%m-%d") for d in dates]


# Columns actually used by compute_kpis()
NEEDED_VCALLS_COLS = [
    "call_date","call_id","company_id","call_direction","spanish_ind",
    "ib_gross_ind","ib_queue_ind","ibs_net_ind","last_ivr_selection_name",
    # used for marketing_bucket + site_serp derivations
    "ivr_split_name","web_session_id",
]

NEEDED_CCF_COLS = [
    "call_id",
    "call_connected_timestamp","compass_start_time","moving_switching_asked_time",
    "movingSwitchingExtracted",
    "return_caller_confirmation_extracted","return_caller_dtmf_boolean","return_caller_confirmation_response",
    "web_address_confirmation_extracted","web_address_confirmation_dtmf_boolean","web_address_confirmation_response",
    "web_zip_confirm_extracted","web_zip_confirmation_dtmf_boolean","web_zip_confirmation_response",
    "web_dob_name_collected_time",
    "zip_code_collected_time","zip_code_confirm_extracted",
    "ss_address_confirmed_time","web_address_confirmed_time","return_caller_confirmed_time",
    "ivrrouting","ercotMatch",
    "name_collected_time","return_caller_path","dob_collected_time",
    "credit_api_start_time","credit_response_status","isCreditHit",
    "first_workflow_leg_id",
    "dead_air_bot","home_business_response",
    "initial_ivr_vf","texas_nontexas_extracted","home_business_extracted",
]

NEEDED_AC_COLS = [
    "call_id",
    "ib_contact_calls","credit_calls_ind","order_count",
]

NEEDED_O_COLS = [
    "call_id",
    "gcv_v2",
]


def get_data(
    marketing_buckets: Optional[Union[str, Sequence[str]]] = None,
    site_serp: Optional[Union[str, Sequence[str]]] = None,
    company_id: Optional[int] = 25,
    call_direction: Optional[str] = "INBOUND",
    restrict_to_yday_and_prior4: bool = True,
    create_temp_view: bool = True,
    print_summary: bool = True,
) -> DataFrame:
    """
    Faster get_data(): only selects columns required by compute_kpis()
    (plus columns needed to derive marketing_bucket/site_serp).

    Returns a wide calls_full DataFrame and optionally creates TEMP VIEW calls_full.
    """

    # ---------------------------
    # Normalize & validate filters (same behavior as before)
    # ---------------------------
    allowed_buckets = {
        "Natural","Brand-Partner","Generic","Aggregator","Competitor","Utility","PMax","NRG","Other Bucket"
    }

    def _as_list(x):
        if x is None:
            return None
        if isinstance(x, str):
            return [x]
        return list(x)

    mb_list = _as_list(marketing_buckets)
    if mb_list is not None:
        norm_map = {
            "brandpartner": "Brand-Partner",
            "brand-partner": "Brand-Partner",
            "brand_partner": "Brand-Partner",
            "pmax": "PMax",
            "nrg": "NRG",
            "other": "Other Bucket",
            "other bucket": "Other Bucket",
            "utility": "Utility",
            "natural": "Natural",
            "generic": "Generic",
            "aggregator": "Aggregator",
            "competitor": "Competitor",
        }
        mb_list = [norm_map.get(b.strip().lower(), b) for b in mb_list]
        unknown = [b for b in mb_list if b not in allowed_buckets]
        if unknown:
            raise ValueError(f"Unknown marketing_buckets: {unknown}. Allowed: {sorted(allowed_buckets)}")

    ss_list = _as_list(site_serp)
    if ss_list is not None:
        norm_ss = {"site": "Site", "serp": "SERP"}
        ss_list = [norm_ss.get(s.strip().lower(), s) for s in ss_list]
        unknown = [s for s in ss_list if s not in {"Site", "SERP"}]
        if unknown:
            raise ValueError("site_serp must be 'Site', 'SERP', or a list of those values.")

    # ---------------------------
    # Source tables
    # ---------------------------
    base_table = "energy_prod.energy.v_calls"
    ccf_table  = "ai_products_prod.energy.compass_call_fct"
    ac_table   = "energy_prod.energy.v_agent_calls"
    o_table    = "energy_prod.energy.v_orders"

    # ---------------------------
    # CASE expressions (unchanged)
    # ---------------------------
    marketing_bucket_case = """
      CASE
        WHEN c.ivr_split_name IN ('natural_marketingbucket','natural_marketingbucket_serp') THEN 'Natural'
        WHEN c.ivr_split_name IN ('brandpartner_marketingbucket_serp','brandpartner_marketingbucket') THEN 'Brand-Partner'
        WHEN c.ivr_split_name IN ('generic_marketingbucket_serp','generic_marketingbucket') THEN 'Generic'
        WHEN c.ivr_split_name IN ('aggregator_marketingbucket_serp','aggregator_marketingbucket') THEN 'Aggregator'
        WHEN c.ivr_split_name IN ('competitor_marketingbucket_serp','competitor_marketingbucket') THEN 'Competitor'
        WHEN c.ivr_split_name IN ('dereg_utility_check_serp','dereg_utility_check') THEN 'Utility'
        WHEN c.ivr_split_name IN ('pmax_marketingbucket_serp','pmax_marketingbucket') THEN 'PMax'
        WHEN c.ivr_split_name IN ('nrg_bucket_serp','nrg_bucket') THEN 'NRG'
        ELSE 'Other Bucket'
      END
    """

    site_serp_case = """
      CASE
        WHEN c.web_session_id IS NOT NULL THEN 'Site'
        ELSE 'SERP'
      END
    """

    # ---------------------------
    # WHERE (base)
    # ---------------------------
    base_where_clauses = []
    if company_id is not None:
        base_where_clauses.append(f"c.company_id = {int(company_id)}")
    if call_direction is not None:
        safe_call_dir = str(call_direction).replace("'", "''")
        base_where_clauses.append(f"UPPER(c.call_direction) = UPPER('{safe_call_dir}')")

    pulled_dates = None
    if restrict_to_yday_and_prior4:
        pulled_dates = _compute_yday_and_prior4_dates()
        dates_sql = ", ".join([f"'{d}'" for d in pulled_dates])
        base_where_clauses.append(f"c.call_date IN ({dates_sql})")

    base_where_sql = "\n        AND ".join(base_where_clauses) if base_where_clauses else "1=1"

    # post filters (marketing bucket / site_serp)
    post_filters = []
    if mb_list:
        mb_sql = ", ".join([f"'{b}'" for b in mb_list])
        post_filters.append(f"marketing_bucket IN ({mb_sql})")
    if ss_list:
        ss_sql = ", ".join([f"'{s}'" for s in ss_list])
        post_filters.append(f"site_serp IN ({ss_sql})")

    post_where_sql = ""
    if post_filters:
        post_where_sql = "WHERE " + " AND ".join(post_filters)

    # ---------------------------
    # SELECT lists (only required cols)
    # ---------------------------
    # v_calls needed + derived fields
    base_select_cols = ", ".join([f"c.`{col}`" for col in NEEDED_VCALLS_COLS])

    # ccf/ac/o needed, with prefixes to match compute_kpis expectations
    ccf_select_cols = ",\n      ".join(
        [f"ccf.`{col}` AS `ccf_{col}`" for col in NEEDED_CCF_COLS if col != "call_id"]
    )
    ac_select_cols = ",\n      ".join(
        [f"ac.`{col}` AS `ac_{col}`" for col in NEEDED_AC_COLS if col != "call_id"]
    )
    o_select_cols = ",\n      ".join(
        [f"o.`{col}` AS `o_{col}`" for col in NEEDED_O_COLS if col != "call_id"]
    )

    # final projection order
    final_select_parts = [
        # base calls (including ivr_split_name/web_session_id needed for derivations)
        ", ".join([f"c.`{col}`" for col in NEEDED_VCALLS_COLS]),
        "c.marketing_bucket",
        "c.site_serp",
        ccf_select_cols,
        ac_select_cols,
        o_select_cols,
    ]
    final_select_sql = ",\n      ".join([p for p in final_select_parts if p.strip()])

    calls_full_sql = f"""
    WITH base_calls AS (
      SELECT
        {base_select_cols},
        {marketing_bucket_case} AS marketing_bucket,
        {site_serp_case} AS site_serp
      FROM {base_table} c
      WHERE {base_where_sql}
    ),
    filtered_calls AS (
      SELECT *
      FROM base_calls
      {post_where_sql}
    )
    SELECT
      {final_select_sql}
    FROM filtered_calls c
    LEFT JOIN {ccf_table} ccf
      ON c.call_id = ccf.call_id
    LEFT JOIN {ac_table} ac
      ON c.call_id = ac.call_id
    LEFT JOIN {o_table} o
      ON c.call_id = o.call_id
    """

    df = spark.sql(calls_full_sql)

    if create_temp_view:
        df.createOrReplaceTempView("calls_full")

    if print_summary:
        if pulled_dates is not None:
            print(f"get_data(): pulled call_date IN {pulled_dates}")
        else:
            print("get_data(): no call_date restriction applied")
        print(f"get_data(): rows = {df.count()}")
        if create_temp_view:
            print("get_data(): created TEMP VIEW calls_full")

    return df


### Full Funnel Query

In [0]:
from typing import Optional
from pyspark.sql import DataFrame

def compute_kpis(
    calls_df: Optional[DataFrame] = None,
    source_view: str = "calls_full",
    create_temp_view: bool = True,
    debug_print_sql: bool = True,
) -> DataFrame:
    """
    Serverless-friendly compute_kpis where the full KPI SQL is embedded in the function.

    - If calls_df is provided and create_temp_view=True, it will createOrReplaceTempView(source_view).
    - The internal SQL references the given source_view and creates a temp view named out_view.
    - Returns a DataFrame from SELECT * FROM out_view.
    """

    # If caller provided a DataFrame, register it as the source view
    if calls_df is not None and create_temp_view:
        calls_df.createOrReplaceTempView(source_view)

    # Build a safe output view name (unique-ish)
    safe_view = (
        source_view.replace("`", "")
        .replace(".", "_")
        .replace("-", "_")
        .replace(" ", "_")
        .replace("|", "_")
        .replace("/", "_")
    )
    out_view = f"full_funnel_kpis__{safe_view}"

    # Embedded SQL (no date-filtering here; computation expects the source_view to already be limited to relevant dates)
    full_funnel_kpis_sql = f"""
CREATE OR REPLACE TEMP VIEW {out_view} AS
WITH params AS (
  SELECT
    date_sub(current_date(), 1)            AS yday,
    dayofweek(date_sub(current_date(), 1)) AS yday_dow,
    date_format(date_sub(current_date(), 1), 'yyyy-MM-dd') AS yday_label
),

base AS (
  SELECT
    cf.call_date,
    cf.call_id,

    -- v_calls (no prefix in calls_full)
    cf.company_id,
    cf.call_direction,
    cf.spanish_ind,
    cf.ib_gross_ind,
    cf.ib_queue_ind,
    cf.ibs_net_ind,

    -- v_agent_calls (ac_)
    cf.ac_ib_contact_calls,
    cf.ac_credit_calls_ind,
    cf.ac_order_count,

    -- v_orders (o_)
    cf.o_gcv_v2,

    -- Entered Compass + key step flags
    CASE
      WHEN try_cast(cf.ccf_call_connected_timestamp AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_compass_start_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS entered_compass,

    CASE
      WHEN try_cast(cf.ccf_moving_switching_asked_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS moving_switching_asked,

    CASE
      WHEN cf.ccf_movingSwitchingExtracted IS NOT NULL
           AND cf.ccf_movingSwitchingExtracted NOT IN ('0','null') THEN 1
      WHEN cf.ccf_return_caller_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_return_caller_dtmf_boolean = 'dtmf'
               AND cf.ccf_return_caller_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_address_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_web_address_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_address_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_zip_confirm_extracted IN ('Yes','yes')
           OR (cf.ccf_web_zip_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_zip_confirmation_response IN (1,11)) THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS moving_switching_extracted,

    CASE
      WHEN try_cast(cf.ccf_zip_code_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_zip_code_confirm_extracted IN ('Yes','yes') THEN 1
      WHEN cf.ccf_web_address_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_web_address_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_address_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_zip_confirm_extracted IN ('Yes','yes')
           OR (cf.ccf_web_zip_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_zip_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_return_caller_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_return_caller_dtmf_boolean = 'dtmf'
               AND cf.ccf_return_caller_confirmation_response IN (1,11)) THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS any_zip_collected,

    CASE
      WHEN try_cast(cf.ccf_ss_address_confirmed_time AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_web_address_confirmed_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_confirmed_time IN ('Yes','yes') THEN 1
      ELSE 0
    END AS any_address_collected,

    CASE
      WHEN cf.ccf_ivrrouting = 'default'
       AND cf.ccf_ercotMatch = 'true'
       AND (
            try_cast(cf.ccf_ss_address_confirmed_time AS timestamp) IS NOT NULL
         OR try_cast(cf.ccf_web_address_confirmed_time AS timestamp) IS NOT NULL
         OR cf.ccf_return_caller_confirmed_time IN ('Yes','yes')
       )
      THEN 1 ELSE 0
    END AS address_collected_matched,

    CASE
      WHEN try_cast(cf.ccf_name_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_path = 'credit' THEN 1
      ELSE 0
    END AS name_confirmed,

    CASE
      WHEN try_cast(cf.ccf_dob_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_path = 'credit' THEN 1
      ELSE 0
    END AS dob_collected,

    CASE
      WHEN try_cast(cf.ccf_credit_api_start_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS ucc_api_call,

    CASE
      WHEN cf.ccf_credit_response_status = 'Successful' THEN 1
      ELSE 0
    END AS ucc_api_call_successful,

    CASE
      WHEN cf.ccf_isCreditHit = 'true' THEN 1
      WHEN cf.ccf_return_caller_confirmed_time IN ('Yes','yes')
           AND cf.ccf_return_caller_path = 'credit' THEN 1
      ELSE 0
    END AS ucc_credit_hit,

    CASE
      WHEN cf.ccf_first_workflow_leg_id IS NOT NULL THEN 1
      ELSE 0
    END AS compass_completion,

    -- Bot flags
    CASE WHEN cf.ccf_dead_air_bot = '1' THEN 1 ELSE 0 END AS deadair_bot_traffic,

    CASE
      WHEN cf.ccf_home_business_response RLIKE
        '(^|[^0-9])' ||
        '([+]?[[:space:]().-]*1[[:space:]().-]*)?' ||
        '(' ||
          '8[0-9]{{2}}[[:space:]().-]*[0-9]{{3}}[[:space:]().-]*[0-9]{{4}}' ||
          '|' ||
          '8[0-9]{{2}}-[0-9]{{3}}' ||
          '|' ||
          '8[0-9]{{2}}-[0-9]{{4}}' ||
        ')' ||
        '([^0-9]|$)'
      THEN 1 ELSE 0
    END AS phone_number_bot_traffic,

    CASE
      WHEN cf.ccf_home_business_response RLIKE '([Pp])ress[[:space:]]+[0-9]'
      THEN 1 ELSE 0
    END AS press_number_bot_traffic,

    CASE
      WHEN (cf.ccf_dead_air_bot = '1')
        OR (cf.ccf_home_business_response RLIKE
          '(^|[^0-9])' ||
          '([+]?[[:space:]().-]*1[[:space:]().-]*)?' ||
          '(' ||
            '8[0-9]{{2}}[[:space:]().-]*[0-9]{{3}}[[:space:]().-]*[0-9]{{4}}' ||
            '|' ||
            '8[0-9]{{2}}-[0-9]{{3}}' ||
            '|' ||
            '8[0-9]{{2}}-[0-9]{{4}}' ||
          ')' ||
          '([^0-9]|$)')
        OR (cf.ccf_home_business_response RLIKE '([Pp])ress[[:space:]]+[0-9]')
      THEN 1 ELSE 0
    END AS any_bot_call,

    -- IVR completion proxy
    CASE
      WHEN cf.last_ivr_selection_name IN ('TX','TX-1') THEN 1
      WHEN cf.ccf_initial_ivr_vf = 1 AND cf.ccf_texas_nontexas_extracted = 'Texas' THEN 1
      WHEN cf.ccf_initial_ivr_vf = 1
           AND cf.ccf_home_business_extracted = 'residential'
           AND cf.call_date < DATE '2025-09-24' THEN 1
      WHEN try_cast(cf.ccf_moving_switching_asked_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_return_caller_dtmf_boolean = 'dtmf'
               AND cf.ccf_return_caller_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_address_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_web_address_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_address_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_zip_confirm_extracted IN ('Yes','yes')
           OR (cf.ccf_web_zip_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_zip_confirmation_response IN (1,11)) THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS ivr_completed_call,

    -- STEP REACH denominators
    CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END AS reached_entered_compass,
    CASE WHEN entered_compass = 1 AND ivr_completed_call = 1 THEN 1 ELSE 0 END AS reached_moving_switching,
    CASE WHEN entered_compass = 1 AND moving_switching_extracted = 1 THEN 1 ELSE 0 END AS reached_zip_collection,
    CASE WHEN entered_compass = 1 AND any_zip_collected = 1 THEN 1 ELSE 0 END AS reached_address_collection,
    CASE WHEN entered_compass = 1 AND any_address_collected = 1 THEN 1 ELSE 0 END AS reached_ercot_check,
    CASE WHEN entered_compass = 1 AND address_collected_matched = 1 THEN 1 ELSE 0 END AS reached_name_collection,
    CASE WHEN entered_compass = 1 AND name_confirmed = 1 THEN 1 ELSE 0 END AS reached_dob_collection,
    CASE WHEN entered_compass = 1 AND dob_collected = 1 THEN 1 ELSE 0 END AS reached_credit_check,

    -- QUEUE @ STEP
    CASE WHEN entered_compass = 1 AND cf.ib_queue_ind = 1 AND ivr_completed_call = 0 THEN 1 ELSE 0 END AS q_at_initial_question,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND moving_switching_extracted = 0 THEN 1 ELSE 0 END AS q_at_moving_switching,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND moving_switching_extracted = 1 AND any_zip_collected = 0 THEN 1 ELSE 0 END AS q_at_zip_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND any_zip_collected = 1 AND any_address_collected = 0 THEN 1 ELSE 0 END AS q_at_address_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND any_address_collected = 1 AND address_collected_matched = 0 THEN 1 ELSE 0 END AS q_at_ercot_check,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND address_collected_matched = 1 AND name_confirmed = 0 THEN 1 ELSE 0 END AS q_at_name_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND name_confirmed = 1 AND dob_collected = 0 THEN 1 ELSE 0 END AS q_at_dob_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND dob_collected = 1 AND ucc_credit_hit = 0 THEN 1 ELSE 0 END AS q_at_credit_check_no_hit,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND ucc_credit_hit = 1 THEN 1 ELSE 0 END AS q_with_credit_hit,

    -- BREAKAGE @ STEP
    CASE WHEN entered_compass = 1 AND cf.ib_queue_ind = 0 AND ivr_completed_call = 0 THEN 1 ELSE 0 END AS b_at_initial_question,
    CASE WHEN cf.ib_queue_ind = 0 AND moving_switching_extracted = 0 AND ivr_completed_call = 1 THEN 1 ELSE 0 END AS b_at_moving_switching,
    CASE WHEN cf.ib_queue_ind = 0 AND moving_switching_extracted = 1
              AND try_cast(cf.ccf_moving_switching_asked_time AS timestamp) IS NOT NULL
              AND any_zip_collected = 0
         THEN 1 ELSE 0 END AS b_at_zip_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND any_zip_collected = 1 AND any_address_collected = 0 THEN 1 ELSE 0 END AS b_at_address_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND any_address_collected = 1 AND address_collected_matched = 0 THEN 1 ELSE 0 END AS b_at_ercot_check,
    CASE WHEN cf.ib_queue_ind = 0 AND address_collected_matched = 1 AND name_confirmed = 0 THEN 1 ELSE 0 END AS b_at_name_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND name_confirmed = 1 AND dob_collected = 0 THEN 1 ELSE 0 END AS b_at_dob_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND dob_collected = 1 AND ucc_credit_hit = 0 THEN 1 ELSE 0 END AS b_at_credit_check_no_hit,
    CASE WHEN cf.ib_queue_ind = 0 AND ucc_credit_hit = 1 THEN 1 ELSE 0 END AS b_with_credit_hit

  FROM {source_view} cf
  CROSS JOIN params p
  WHERE cf.call_date >= date_sub(p.yday, 60)
    AND cf.call_date <= p.yday
    AND dayofweek(cf.call_date) = p.yday_dow
    AND cf.spanish_ind = 0
    AND cf.company_id = 25
    AND cf.call_direction IN ('Inbound','INBOUND')
),

prior4_dates AS (
  SELECT call_date
  FROM (
    SELECT
      call_date,
      dense_rank() OVER (ORDER BY call_date DESC) AS dr
    FROM (SELECT DISTINCT call_date FROM base)
    CROSS JOIN params p
    WHERE call_date < p.yday
  ) t
  WHERE dr <= 4
),

scoped AS (
  SELECT
    b.*,
    CASE
      WHEN b.call_date = p.yday THEN 'yesterday'
      WHEN b.call_date IN (SELECT call_date FROM prior4_dates) THEN 'prior4'
      ELSE 'ignore'
    END AS period
  FROM base b
  CROSS JOIN params p
),

daily AS (
  SELECT
    call_date,
    period,

    -- Core volumes
    SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END) AS gross_calls,

    SUM(CASE WHEN ib_gross_ind = 1 THEN entered_compass ELSE 0 END) AS entered_compass_calls,
    try_divide(
      SUM(CASE WHEN ib_gross_ind = 1 THEN entered_compass ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS entered_compass_rate,

    try_divide(
      SUM(compass_completion),
      NULLIF(SUM(entered_compass), 0)
    ) AS compass_completion_rate,

    -- Bot KPIs (Compass-scoped)
    try_divide(
      SUM(CASE WHEN entered_compass = 1 AND any_bot_call = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END), 0)
    ) AS any_bot_call_rate_compass,

    ( SUM(CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END)
      - SUM(CASE WHEN entered_compass = 1 AND any_bot_call = 1 THEN 1 ELSE 0 END)
    ) AS non_bot_compass_calls,

    -- Queue metrics
    SUM(CASE WHEN ib_queue_ind = 1 THEN 1 ELSE 0 END) AS queue_calls,
    try_divide(
      SUM(CASE WHEN ib_queue_ind = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS queue_to_gross_rate,

    try_divide(
      SUM(CASE WHEN entered_compass = 1 THEN ivr_completed_call ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_queue_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS queue_to_ivr_rate,

    try_divide(
      SUM(CASE WHEN ib_queue_ind = 1 AND ibs_net_ind = 0 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS abandonment_rate,

    -- Net + downstream rates
    SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END) AS net_calls,

    try_divide(
      SUM(CASE WHEN ibs_net_ind = 1 AND ac_ib_contact_calls = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS contact_rate,

    try_divide(
      SUM(CASE WHEN ibs_net_ind = 1 AND ac_credit_calls_ind = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS credit_rate,

    try_divide(
      SUM(CASE WHEN ibs_net_ind = 1 AND ac_order_count = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS net_conversion_rate,

    SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END) AS orders,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS gross_conversion_rate,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END),
      NULLIF(
        ( SUM(CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END)
          - SUM(CASE WHEN entered_compass = 1 AND any_bot_call = 1 THEN 1 ELSE 0 END)
        ),
        0
      )
    ) AS transactional_conversion_rate,

    -- UCC operational rates
    try_divide(SUM(ucc_api_call),            NULLIF(SUM(entered_compass), 0)) AS ucc_api_call_rate,
    try_divide(SUM(ucc_api_call_successful), NULLIF(SUM(ucc_api_call), 0))    AS ucc_api_success_given_call_rate,
    try_divide(SUM(ucc_credit_hit),          NULLIF(SUM(ucc_api_call_successful), 0)) AS ucc_hit_given_success_rate,

    -- Step loss rates
    try_divide(SUM(q_at_initial_question), NULLIF(SUM(reached_entered_compass), 0)) AS q_rate_initial_question,
    try_divide(SUM(b_at_initial_question), NULLIF(SUM(reached_entered_compass), 0)) AS b_rate_initial_question,

    try_divide(SUM(q_at_moving_switching), NULLIF(SUM(reached_moving_switching), 0)) AS q_rate_moving_switching,
    try_divide(SUM(b_at_moving_switching), NULLIF(SUM(reached_moving_switching), 0)) AS b_rate_moving_switching,

    try_divide(SUM(q_at_zip_collection), NULLIF(SUM(reached_zip_collection), 0)) AS q_rate_zip_collection,
    try_divide(SUM(b_at_zip_collection), NULLIF(SUM(reached_zip_collection), 0)) AS b_rate_zip_collection,

    try_divide(SUM(q_at_address_collection), NULLIF(SUM(reached_address_collection), 0)) AS q_rate_address_collection,
    try_divide(SUM(b_at_address_collection), NULLIF(SUM(reached_address_collection), 0)) AS b_rate_address_collection,

    try_divide(SUM(q_at_ercot_check), NULLIF(SUM(reached_ercot_check), 0)) AS q_rate_ercot_check,
    try_divide(SUM(b_at_ercot_check), NULLIF(SUM(reached_ercot_check), 0)) AS b_rate_ercot_check,

    try_divide(SUM(q_at_name_collection), NULLIF(SUM(reached_name_collection), 0)) AS q_rate_name_collection,
    try_divide(SUM(b_at_name_collection), NULLIF(SUM(reached_name_collection), 0)) AS b_rate_name_collection,

    try_divide(SUM(q_at_dob_collection), NULLIF(SUM(reached_dob_collection), 0)) AS q_rate_dob_collection,
    try_divide(SUM(b_at_dob_collection), NULLIF(SUM(reached_dob_collection), 0)) AS b_rate_dob_collection,

    try_divide(SUM(q_at_credit_check_no_hit), NULLIF(SUM(reached_credit_check), 0)) AS q_rate_credit_check_no_hit,
    try_divide(SUM(b_at_credit_check_no_hit), NULLIF(SUM(reached_credit_check), 0)) AS b_rate_credit_check_no_hit,

    try_divide(SUM(q_with_credit_hit), NULLIF(SUM(reached_credit_check), 0)) AS q_rate_with_credit_hit,
    try_divide(SUM(b_with_credit_hit), NULLIF(SUM(reached_credit_check), 0)) AS b_rate_with_credit_hit,

    -- Step counts (unchanged)
    SUM(reached_entered_compass) AS reached_entered_compass_calls,
    SUM(reached_moving_switching) AS reached_moving_switching_calls,
    SUM(reached_zip_collection) AS reached_zip_collection_calls,
    SUM(reached_address_collection) AS reached_address_collection_calls,
    SUM(reached_ercot_check) AS reached_ercot_check_calls,
    SUM(reached_name_collection) AS reached_name_collection_calls,
    SUM(reached_dob_collection) AS reached_dob_collection_calls,
    SUM(reached_credit_check) AS reached_credit_check_calls,

    SUM(q_at_initial_question) AS q_at_initial_question_calls,
    SUM(b_at_initial_question) AS b_at_initial_question_calls,
    SUM(q_at_moving_switching) AS q_at_moving_switching_calls,
    SUM(b_at_moving_switching) AS b_at_moving_switching_calls,
    SUM(q_at_zip_collection) AS q_at_zip_collection_calls,
    SUM(b_at_zip_collection) AS b_at_zip_collection_calls,
    SUM(q_at_address_collection) AS q_at_address_collection_calls,
    SUM(b_at_address_collection) AS b_at_address_collection_calls,
    SUM(q_at_ercot_check) AS q_at_ercot_check_calls,
    SUM(b_at_ercot_check) AS b_at_ercot_check_calls,
    SUM(q_at_name_collection) AS q_at_name_collection_calls,
    SUM(b_at_name_collection) AS b_at_name_collection_calls,
    SUM(q_at_dob_collection) AS q_at_dob_collection_calls,
    SUM(b_at_dob_collection) AS b_at_dob_collection_calls,
    SUM(q_at_credit_check_no_hit) AS q_at_credit_check_no_hit_calls,
    SUM(b_at_credit_check_no_hit) AS b_at_credit_check_no_hit_calls,
    SUM(q_with_credit_hit) AS q_with_credit_hit_calls,
    SUM(b_with_credit_hit) AS b_with_credit_hit_calls,

    SUM(CASE WHEN entered_compass = 1 THEN ivr_completed_call ELSE 0 END) AS ivr_completed_calls,

    SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END) AS total_revenue,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END),
      NULLIF(SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END), 0)
    ) AS revenue_per_order,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS revenue_per_net_call,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS revenue_per_gross_call

  FROM scoped
  WHERE period IN ('yesterday','prior4')
  GROUP BY call_date, period
),

period_agg AS (
  SELECT
    MAX(p.yday_label) AS yday_label,

    AVG(CASE WHEN period='prior4' THEN CAST(gross_calls AS DOUBLE) END) AS p4_gross_calls,
    AVG(CASE WHEN period='prior4' THEN entered_compass_rate END) AS p4_entered_compass_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(entered_compass_calls AS DOUBLE) END) AS p4_entered_compass_calls,
    AVG(CASE WHEN period='prior4' THEN compass_completion_rate END) AS p4_compass_completion_rate,

    AVG(CASE WHEN period='prior4' THEN any_bot_call_rate_compass END) AS p4_any_bot_call_rate_compass,
    AVG(CASE WHEN period='prior4' THEN CAST(non_bot_compass_calls AS DOUBLE) END) AS p4_non_bot_compass_calls,

    AVG(CASE WHEN period='prior4' THEN queue_to_gross_rate END) AS p4_queue_to_gross_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(queue_calls AS DOUBLE) END) AS p4_queue_calls,
    AVG(CASE WHEN period='prior4' THEN abandonment_rate END) AS p4_abandonment_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(net_calls AS DOUBLE) END) AS p4_net_calls,

    AVG(CASE WHEN period='prior4' THEN contact_rate END) AS p4_contact_rate,
    AVG(CASE WHEN period='prior4' THEN credit_rate END) AS p4_credit_rate,
    AVG(CASE WHEN period='prior4' THEN net_conversion_rate END) AS p4_net_conversion_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(orders AS DOUBLE) END) AS p4_orders,
    AVG(CASE WHEN period='prior4' THEN gross_conversion_rate END) AS p4_gross_conversion_rate,

    AVG(CASE WHEN period='prior4' THEN ucc_api_call_rate END) AS p4_ucc_api_call_rate,
    AVG(CASE WHEN period='prior4' THEN ucc_api_success_given_call_rate END) AS p4_ucc_api_success_given_call_rate,
    AVG(CASE WHEN period='prior4' THEN ucc_hit_given_success_rate END) AS p4_ucc_hit_given_success_rate,

    AVG(CASE WHEN period='prior4' THEN q_rate_initial_question END) AS p4_q_rate_initial_question,
    AVG(CASE WHEN period='prior4' THEN b_rate_initial_question END) AS p4_b_rate_initial_question,

    AVG(CASE WHEN period='prior4' THEN q_rate_moving_switching END) AS p4_q_rate_moving_switching,
    AVG(CASE WHEN period='prior4' THEN b_rate_moving_switching END) AS p4_b_rate_moving_switching,

    AVG(CASE WHEN period='prior4' THEN q_rate_zip_collection END) AS p4_q_rate_zip_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_zip_collection END) AS p4_b_rate_zip_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_address_collection END) AS p4_q_rate_address_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_address_collection END) AS p4_b_rate_address_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_ercot_check END) AS p4_q_rate_ercot_check,
    AVG(CASE WHEN period='prior4' THEN b_rate_ercot_check END) AS p4_b_rate_ercot_check,

    AVG(CASE WHEN period='prior4' THEN q_rate_name_collection END) AS p4_q_rate_name_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_name_collection END) AS p4_b_rate_name_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_dob_collection END) AS p4_q_rate_dob_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_dob_collection END) AS p4_b_rate_dob_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_credit_check_no_hit END) AS p4_q_rate_credit_check_no_hit,
    AVG(CASE WHEN period='prior4' THEN b_rate_credit_check_no_hit END) AS p4_b_rate_credit_check_no_hit,

    AVG(CASE WHEN period='prior4' THEN q_rate_with_credit_hit END) AS p4_q_rate_with_credit_hit,
    AVG(CASE WHEN period='prior4' THEN b_rate_with_credit_hit END) AS p4_b_rate_with_credit_hit,

    -- step counts prior4
    AVG(CASE WHEN period='prior4' THEN CAST(reached_entered_compass_calls AS DOUBLE) END)      AS p4_reached_entered_compass_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_moving_switching_calls AS DOUBLE) END)     AS p4_reached_moving_switching_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_zip_collection_calls AS DOUBLE) END)       AS p4_reached_zip_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_address_collection_calls AS DOUBLE) END)   AS p4_reached_address_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_ercot_check_calls AS DOUBLE) END)          AS p4_reached_ercot_check_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_name_collection_calls AS DOUBLE) END)      AS p4_reached_name_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_dob_collection_calls AS DOUBLE) END)       AS p4_reached_dob_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_credit_check_calls AS DOUBLE) END)         AS p4_reached_credit_check_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_initial_question_calls AS DOUBLE) END)        AS p4_q_at_initial_question_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_initial_question_calls AS DOUBLE) END)        AS p4_b_at_initial_question_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_moving_switching_calls AS DOUBLE) END)        AS p4_q_at_moving_switching_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_moving_switching_calls AS DOUBLE) END)        AS p4_b_at_moving_switching_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_zip_collection_calls AS DOUBLE) END)          AS p4_q_at_zip_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_zip_collection_calls AS DOUBLE) END)          AS p4_b_at_zip_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_address_collection_calls AS DOUBLE) END)      AS p4_q_at_address_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_address_collection_calls AS DOUBLE) END)      AS p4_b_at_address_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_ercot_check_calls AS DOUBLE) END)             AS p4_q_at_ercot_check_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_ercot_check_calls AS DOUBLE) END)             AS p4_b_at_ercot_check_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_name_collection_calls AS DOUBLE) END)         AS p4_q_at_name_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_name_collection_calls AS DOUBLE) END)         AS p4_b_at_name_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_dob_collection_calls AS DOUBLE) END)          AS p4_q_at_dob_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_dob_collection_calls AS DOUBLE) END)          AS p4_b_at_dob_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_credit_check_no_hit_calls AS DOUBLE) END)     AS p4_q_at_credit_check_no_hit_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_credit_check_no_hit_calls AS DOUBLE) END)     AS p4_b_at_credit_check_no_hit_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_with_credit_hit_calls AS DOUBLE) END)            AS p4_q_with_credit_hit_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_with_credit_hit_calls AS DOUBLE) END)            AS p4_b_with_credit_hit_calls,

    AVG(CASE WHEN period='prior4' THEN queue_to_ivr_rate END) AS p4_queue_to_ivr_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(ivr_completed_calls AS DOUBLE) END) AS p4_ivr_completed_calls,

    AVG(CASE WHEN period='prior4' THEN transactional_conversion_rate END) AS p4_transactional_conversion_rate,

    AVG(CASE WHEN period='prior4' THEN CAST(total_revenue AS DOUBLE) END) AS p4_total_revenue,
    AVG(CASE WHEN period='prior4' THEN revenue_per_order END) AS p4_revenue_per_order,
    AVG(CASE WHEN period='prior4' THEN revenue_per_net_call END) AS p4_revenue_per_net_call,
    AVG(CASE WHEN period='prior4' THEN revenue_per_gross_call END) AS p4_revenue_per_gross_call,

    -- yesterday values
    MAX(CASE WHEN period='yesterday' THEN CAST(gross_calls AS DOUBLE) END) AS y_gross_calls,
    MAX(CASE WHEN period='yesterday' THEN entered_compass_rate END) AS y_entered_compass_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(entered_compass_calls AS DOUBLE) END) AS y_entered_compass_calls,
    MAX(CASE WHEN period='yesterday' THEN compass_completion_rate END) AS y_compass_completion_rate,

    MAX(CASE WHEN period='yesterday' THEN any_bot_call_rate_compass END) AS y_any_bot_call_rate_compass,
    MAX(CASE WHEN period='yesterday' THEN CAST(non_bot_compass_calls AS DOUBLE) END) AS y_non_bot_compass_calls,

    MAX(CASE WHEN period='yesterday' THEN queue_to_gross_rate END) AS y_queue_to_gross_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(queue_calls AS DOUBLE) END) AS y_queue_calls,
    MAX(CASE WHEN period='yesterday' THEN abandonment_rate END) AS y_abandonment_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(net_calls AS DOUBLE) END) AS y_net_calls,

    MAX(CASE WHEN period='yesterday' THEN contact_rate END) AS y_contact_rate,
    MAX(CASE WHEN period='yesterday' THEN credit_rate END) AS y_credit_rate,
    MAX(CASE WHEN period='yesterday' THEN net_conversion_rate END) AS y_net_conversion_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(orders AS DOUBLE) END) AS y_orders,
    MAX(CASE WHEN period='yesterday' THEN gross_conversion_rate END) AS y_gross_conversion_rate,

    MAX(CASE WHEN period='yesterday' THEN ucc_api_call_rate END) AS y_ucc_api_call_rate,
    MAX(CASE WHEN period='yesterday' THEN ucc_api_success_given_call_rate END) AS y_ucc_api_success_given_call_rate,
    MAX(CASE WHEN period='yesterday' THEN ucc_hit_given_success_rate END) AS y_ucc_hit_given_success_rate,

    MAX(CASE WHEN period='yesterday' THEN q_rate_initial_question END) AS y_q_rate_initial_question,
    MAX(CASE WHEN period='yesterday' THEN b_rate_initial_question END) AS y_b_rate_initial_question,

    MAX(CASE WHEN period='yesterday' THEN q_rate_moving_switching END) AS y_q_rate_moving_switching,
    MAX(CASE WHEN period='yesterday' THEN b_rate_moving_switching END) AS y_b_rate_moving_switching,

    MAX(CASE WHEN period='yesterday' THEN q_rate_zip_collection END) AS y_q_rate_zip_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_zip_collection END) AS y_b_rate_zip_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_address_collection END) AS y_q_rate_address_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_address_collection END) AS y_b_rate_address_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_ercot_check END) AS y_q_rate_ercot_check,
    MAX(CASE WHEN period='yesterday' THEN b_rate_ercot_check END) AS y_b_rate_ercot_check,

    MAX(CASE WHEN period='yesterday' THEN q_rate_name_collection END) AS y_q_rate_name_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_name_collection END) AS y_b_rate_name_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_dob_collection END) AS y_q_rate_dob_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_dob_collection END) AS y_b_rate_dob_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_credit_check_no_hit END) AS y_q_rate_credit_check_no_hit,
    MAX(CASE WHEN period='yesterday' THEN b_rate_credit_check_no_hit END) AS y_b_rate_credit_check_no_hit,

    MAX(CASE WHEN period='yesterday' THEN q_rate_with_credit_hit END) AS y_q_rate_with_credit_hit,
    MAX(CASE WHEN period='yesterday' THEN b_rate_with_credit_hit END) AS y_b_rate_with_credit_hit,

    MAX(CASE WHEN period='yesterday' THEN CAST(total_revenue AS DOUBLE) END) AS y_total_revenue,
    MAX(CASE WHEN period='yesterday' THEN revenue_per_order END)            AS y_revenue_per_order,
    MAX(CASE WHEN period='yesterday' THEN revenue_per_net_call END)         AS y_revenue_per_net_call,
    MAX(CASE WHEN period='yesterday' THEN revenue_per_gross_call END)       AS y_revenue_per_gross_call,

    MAX(CASE WHEN period='yesterday' THEN queue_to_ivr_rate END) AS y_queue_to_ivr_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(ivr_completed_calls AS DOUBLE) END) AS y_ivr_completed_calls,
    MAX(CASE WHEN period='yesterday' THEN transactional_conversion_rate END) AS y_transactional_conversion_rate

  FROM daily
  CROSS JOIN params p
),

final AS (

  /* -------------------------
     VOLUME + ENTRY
     ------------------------- */

  SELECT  1 AS sort_key, 'Gross Calls' AS KPI,
          p4_gross_calls AS P4WA, y_gross_calls AS y_value,
          try_divide(y_gross_calls - p4_gross_calls, NULLIF(p4_gross_calls,0)) AS Delta,
          (y_gross_calls - p4_gross_calls) AS Call_Count_Delta
  FROM period_agg

  UNION ALL SELECT  2, 'Entered Compass Calls (Count)',
          p4_entered_compass_calls, y_entered_compass_calls,
          try_divide(y_entered_compass_calls - p4_entered_compass_calls, NULLIF(p4_entered_compass_calls,0)),
          (y_entered_compass_calls - p4_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  3, 'Entered Compass Rate',
          p4_entered_compass_rate, y_entered_compass_rate,
          try_divide(y_entered_compass_rate - p4_entered_compass_rate, NULLIF(p4_entered_compass_rate,0)),
          (y_entered_compass_calls - p4_entered_compass_calls)
  FROM period_agg


  /* -------------------------
     BOT
     ------------------------- */

  UNION ALL SELECT  4, 'Any Bot Call Rate (Compass)',
          p4_any_bot_call_rate_compass, y_any_bot_call_rate_compass,
          try_divide(y_any_bot_call_rate_compass - p4_any_bot_call_rate_compass, NULLIF(p4_any_bot_call_rate_compass,0)),
          (y_entered_compass_calls - p4_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  5, 'Non-Bot Compass Calls (Count)',
          p4_non_bot_compass_calls, y_non_bot_compass_calls,
          try_divide(y_non_bot_compass_calls - p4_non_bot_compass_calls, NULLIF(p4_non_bot_compass_calls,0)),
          (y_non_bot_compass_calls - p4_non_bot_compass_calls)
  FROM period_agg


  /* -------------------------
     STEP-LEVEL QUEUE RATES
     ------------------------- */

  UNION ALL SELECT  30, 'Queue Rate – Initial Question',
          p4_q_rate_initial_question, y_q_rate_initial_question,
          try_divide(y_q_rate_initial_question - p4_q_rate_initial_question, NULLIF(p4_q_rate_initial_question,0)),
          (y_entered_compass_calls - p4_reached_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  31, 'Queue Rate – Moving / Switching',
          p4_q_rate_moving_switching, y_q_rate_moving_switching,
          try_divide(y_q_rate_moving_switching - p4_q_rate_moving_switching, NULLIF(p4_q_rate_moving_switching,0)),
          (y_entered_compass_calls - p4_reached_moving_switching_calls)
  FROM period_agg

  UNION ALL SELECT  32, 'Queue Rate – ZIP Collection',
          p4_q_rate_zip_collection, y_q_rate_zip_collection,
          try_divide(y_q_rate_zip_collection - p4_q_rate_zip_collection, NULLIF(p4_q_rate_zip_collection,0)),
          (y_entered_compass_calls - p4_reached_zip_collection_calls)
  FROM period_agg

  UNION ALL SELECT  33, 'Queue Rate – Address Collection',
          p4_q_rate_address_collection, y_q_rate_address_collection,
          try_divide(y_q_rate_address_collection - p4_q_rate_address_collection, NULLIF(p4_q_rate_address_collection,0)),
          (y_entered_compass_calls - p4_reached_address_collection_calls)
  FROM period_agg

  UNION ALL SELECT  34, 'Queue Rate – ERCOT Check',
          p4_q_rate_ercot_check, y_q_rate_ercot_check,
          try_divide(y_q_rate_ercot_check - p4_q_rate_ercot_check, NULLIF(p4_q_rate_ercot_check,0)),
          (y_entered_compass_calls - p4_reached_ercot_check_calls)
  FROM period_agg

  UNION ALL SELECT  35, 'Queue Rate – Name Collection',
          p4_q_rate_name_collection, y_q_rate_name_collection,
          try_divide(y_q_rate_name_collection - p4_q_rate_name_collection, NULLIF(p4_q_rate_name_collection,0)),
          (y_entered_compass_calls - p4_reached_name_collection_calls)
  FROM period_agg

  UNION ALL SELECT  36, 'Queue Rate – DOB Collection',
          p4_q_rate_dob_collection, y_q_rate_dob_collection,
          try_divide(y_q_rate_dob_collection - p4_q_rate_dob_collection, NULLIF(p4_q_rate_dob_collection,0)),
          (y_entered_compass_calls - p4_reached_dob_collection_calls)
  FROM period_agg

  UNION ALL SELECT  37, 'Queue Rate – Credit Check (No Hit)',
          p4_q_rate_credit_check_no_hit, y_q_rate_credit_check_no_hit,
          try_divide(y_q_rate_credit_check_no_hit - p4_q_rate_credit_check_no_hit, NULLIF(p4_q_rate_credit_check_no_hit,0)),
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg

  UNION ALL SELECT  38, 'Queue Rate – Credit Check (Hit)',
          p4_q_rate_with_credit_hit, y_q_rate_with_credit_hit,
          try_divide(y_q_rate_with_credit_hit - p4_q_rate_with_credit_hit, NULLIF(p4_q_rate_with_credit_hit,0)),
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg


  /* -------------------------
     STEP-LEVEL BREAKAGE RATES
     ------------------------- */

  UNION ALL SELECT  40, 'Breakage Rate – Initial Question',
          p4_b_rate_initial_question, y_b_rate_initial_question,
          try_divide(y_b_rate_initial_question - p4_b_rate_initial_question, NULLIF(p4_b_rate_initial_question,0)),
          (y_entered_compass_calls - p4_reached_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  41, 'Breakage Rate – Moving / Switching',
          p4_b_rate_moving_switching, y_b_rate_moving_switching,
          try_divide(y_b_rate_moving_switching - p4_b_rate_moving_switching, NULLIF(p4_b_rate_moving_switching,0)),
          (y_entered_compass_calls - p4_reached_moving_switching_calls)
  FROM period_agg

  UNION ALL SELECT  42, 'Breakage Rate – ZIP Collection',
          p4_b_rate_zip_collection, y_b_rate_zip_collection,
          try_divide(y_b_rate_zip_collection - p4_b_rate_zip_collection, NULLIF(p4_b_rate_zip_collection,0)),
          (y_entered_compass_calls - p4_reached_zip_collection_calls)
  FROM period_agg

  UNION ALL SELECT  43, 'Breakage Rate – Address Collection',
          p4_b_rate_address_collection, y_b_rate_address_collection,
          try_divide(y_b_rate_address_collection - p4_b_rate_address_collection, NULLIF(p4_b_rate_address_collection,0)),
          (y_entered_compass_calls - p4_reached_address_collection_calls)
  FROM period_agg

  UNION ALL SELECT  44, 'Breakage Rate – ERCOT Check',
          p4_b_rate_ercot_check, y_b_rate_ercot_check,
          try_divide(y_b_rate_ercot_check - p4_b_rate_ercot_check, NULLIF(p4_b_rate_ercot_check,0)),
          (y_entered_compass_calls - p4_reached_ercot_check_calls)
  FROM period_agg

  UNION ALL SELECT  45, 'Breakage Rate – Name Collection',
          p4_b_rate_name_collection, y_b_rate_name_collection,
          try_divide(y_b_rate_name_collection - p4_b_rate_name_collection, NULLIF(p4_b_rate_name_collection,0)),
          (y_entered_compass_calls - p4_reached_name_collection_calls)
  FROM period_agg

  UNION ALL SELECT  46, 'Breakage Rate – DOB Collection',
          p4_b_rate_dob_collection, y_b_rate_dob_collection,
          try_divide(y_b_rate_dob_collection - p4_b_rate_dob_collection, NULLIF(p4_b_rate_dob_collection,0)),
          (y_entered_compass_calls - p4_reached_dob_collection_calls)
  FROM period_agg

  UNION ALL SELECT  47, 'Breakage Rate – Credit Check (No Hit)',
          p4_b_rate_credit_check_no_hit, y_b_rate_credit_check_no_hit,
          try_divide(y_b_rate_credit_check_no_hit - p4_b_rate_credit_check_no_hit, NULLIF(p4_b_rate_credit_check_no_hit,0)),
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg

  UNION ALL SELECT  48, 'Breakage Rate – Credit Check (Hit)',
          p4_b_rate_with_credit_hit, y_b_rate_with_credit_hit,
          try_divide(y_b_rate_with_credit_hit - p4_b_rate_with_credit_hit, NULLIF(p4_b_rate_with_credit_hit,0)),
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg


  /* -------------------------
     IVR COMPLETION (place before Queue / Abandon per request)
     ------------------------- */

  UNION ALL SELECT  49, 'Passed Initial Question',
          p4_ivr_completed_calls, y_ivr_completed_calls,
          try_divide(y_ivr_completed_calls - p4_ivr_completed_calls, NULLIF(p4_ivr_completed_calls,0)),
          (y_ivr_completed_calls - p4_ivr_completed_calls)
  FROM period_agg

  UNION ALL SELECT  49.1, 'Queue to Initial Question Passed Rate',
          p4_queue_to_ivr_rate, y_queue_to_ivr_rate,
          try_divide(y_queue_to_ivr_rate - p4_queue_to_ivr_rate, NULLIF(p4_queue_to_ivr_rate,0)),
          (y_queue_calls - p4_queue_calls)
  FROM period_agg


  /* -------------------------
     QUEUE / ABANDON
     ------------------------- */

  UNION ALL SELECT  50, 'Queue Calls',
          p4_queue_calls, y_queue_calls,
          try_divide(y_queue_calls - p4_queue_calls, NULLIF(p4_queue_calls,0)),
          (y_queue_calls - p4_queue_calls)
  FROM period_agg

  UNION ALL SELECT  51, 'Queue to Gross Rate',
          p4_queue_to_gross_rate, y_queue_to_gross_rate,
          try_divide(y_queue_to_gross_rate - p4_queue_to_gross_rate, NULLIF(p4_queue_to_gross_rate,0)),
          (y_gross_calls - p4_gross_calls)
  FROM period_agg

  UNION ALL SELECT  52, 'Abandonment Rate - Hung Up in Queue',
          p4_abandonment_rate, y_abandonment_rate,
          try_divide(y_abandonment_rate - p4_abandonment_rate, NULLIF(p4_abandonment_rate,0)),
          (y_queue_calls - p4_queue_calls)
  FROM period_agg


  /* -------------------------
     NET / SALES
     ------------------------- */

  UNION ALL SELECT  60, 'Net Calls',
          p4_net_calls, y_net_calls,
          try_divide(y_net_calls - p4_net_calls, NULLIF(p4_net_calls,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  61, 'Contact Calls/Net Calls',
          p4_contact_rate, y_contact_rate,
          try_divide(y_contact_rate - p4_contact_rate, NULLIF(p4_contact_rate,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  62, 'Credit Run/Net Calls',
          p4_credit_rate, y_credit_rate,
          try_divide(y_credit_rate - p4_credit_rate, NULLIF(p4_credit_rate,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  63, 'Net Conversion Rate',
          p4_net_conversion_rate, y_net_conversion_rate,
          try_divide(y_net_conversion_rate - p4_net_conversion_rate, NULLIF(p4_net_conversion_rate,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  64, 'Orders',
          p4_orders, y_orders,
          try_divide(y_orders - p4_orders, NULLIF(p4_orders,0)),
          (y_orders - p4_orders)
  FROM period_agg

  -- transactional conversion rate BEFORE gross conversion rate (per request)
  UNION ALL SELECT  64.5, 'Non-Bot Compass Call Conversion Rate',
          p4_transactional_conversion_rate, y_transactional_conversion_rate,
          try_divide(y_transactional_conversion_rate - p4_transactional_conversion_rate, NULLIF(p4_transactional_conversion_rate,0)),
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  65, 'Gross Conversion Rate',
          p4_gross_conversion_rate, y_gross_conversion_rate,
          try_divide(y_gross_conversion_rate - p4_gross_conversion_rate, NULLIF(p4_gross_conversion_rate,0)),
          (y_gross_calls - p4_gross_calls)
  FROM period_agg


  /* -------------------------
     REVENUE (at the end)
     ------------------------- */

  UNION ALL SELECT  90, 'Total Revenue',
          p4_total_revenue, y_total_revenue,
          try_divide(y_total_revenue - p4_total_revenue, NULLIF(p4_total_revenue,0)),
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  91, 'Revenue per Order',
          p4_revenue_per_order, y_revenue_per_order,
          try_divide(y_revenue_per_order - p4_revenue_per_order, NULLIF(p4_revenue_per_order,0)),
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  92, 'Revenue per Net Call',
          p4_revenue_per_net_call, y_revenue_per_net_call,
          try_divide(y_revenue_per_net_call - p4_revenue_per_net_call, NULLIF(p4_revenue_per_net_call,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  93, 'Revenue per Gross Call',
          p4_revenue_per_gross_call, y_revenue_per_gross_call,
          try_divide(y_revenue_per_gross_call - p4_revenue_per_gross_call, NULLIF(p4_revenue_per_gross_call,0)),
          (y_gross_calls - p4_gross_calls)
  FROM period_agg

)

SELECT
  sort_key,
  KPI,
  P4WA,
  y_value AS `Yesterday`,
  Delta,
  Call_Count_Delta
FROM final
ORDER BY sort_key
;

"""

    # Optionally preview the SQL (first N chars)
    if debug_print_sql:
        print(f"--- compute_kpis() will create/replace temp view: {out_view} ---")
        # print a short preview so notebook output remains readable
        print(full_funnel_kpis_sql[:3000])
        print("--- end SQL preview ---\n")

    try:
        # Execute the embedded SQL which must create the {out_view} temp view
        spark.sql(full_funnel_kpis_sql)

        # Return the results
        return spark.sql(f"SELECT * FROM {out_view}")

    except Exception as e:
        print("ERROR running compute_kpis():", type(e).__name__, str(e))
        try:
            print("\n--- SQL (first 8000 chars) ---")
            print(full_funnel_kpis_sql[:8000])
            print("\n--- SQL (last 8000 chars) ---")
            print(full_funnel_kpis_sql[-8000:])
        except Exception:
            pass
        raise


### Converting SQL Output to Structured JSON

In [0]:
import re
from typing import Any, Dict, List, Optional, Union

# -----------------------------
# helpers (unchanged)
# -----------------------------
def _to_snake(s: str) -> str:
    s = s.strip().lower()
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"[/%()]", " ", s)
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def _safe_float(x: Any) -> Optional[float]:
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return None

def _is_breakdown_kpi(kpi_name: str, delim: str = " - ") -> bool:
    norm = kpi_name.replace("–", "-").replace("—", "-")
    return " - " in norm

def _split_breakdown(kpi_name: str):
    norm = kpi_name.replace("–", "-").replace("—", "-")
    parts = [p.strip() for p in norm.split("-", 1)]
    if len(parts) != 2:
        return (kpi_name.strip(), None)
    return (parts[0].strip(), parts[1].strip())


# -----------------------------
# UPDATED VERSION (no per-step call counts)
# -----------------------------
def kpi_tables_to_json(
    names: List[str],
    tables: List[Union[str, Any]],  # str table/view name OR Spark DataFrame
    *,
    kpi_col: str = "KPI",
    current_col: str = "Yesterday",
    baseline_col: str = "P4WA",
    delta_col: str = "Delta",
    call_delta_col: str = "Call_Count_Delta",
    breakdown_reason_key: str = "reason",
    prefer_precomputed_delta: bool = True,
    # Speed / safety knobs
    max_collect_rows: int = 2000,                 # fail fast if a KPI DF is accidentally huge
    use_collect_instead_of_pandas: bool = True,   # avoid toPandas overhead
) -> Dict[str, Any]:
    """
    Updated to match compute_kpis output that REMOVED per-step call count fields:
      - Removed reading/output of Yesterday_calls / P4WA_calls.
      - Keeps Call_Count_Delta when present (as calls_delta).
    """
    if len(names) != len(tables):
        raise ValueError(f"names and tables must be same length. Got {len(names)} vs {len(tables)}")

    out: Dict[str, Any] = {}

    for section_name, tbl in zip(names, tables):
        sdf = spark.table(tbl) if isinstance(tbl, str) else tbl

        cols = set(sdf.columns)
        required = {kpi_col, current_col, baseline_col}
        missing = required - cols
        if missing:
            raise ValueError(f"Section '{section_name}' missing required columns: {sorted(missing)}")

        # Only pull what we actually need
        wanted = [kpi_col, current_col, baseline_col]
        if delta_col in cols:
            wanted.append(delta_col)
        if call_delta_col in cols:
            wanted.append(call_delta_col)

        sdf_small = sdf.select(*wanted)

        # Guardrail: KPI outputs should be small; avoid accidental "collect millions"
        n = sdf_small.count()
        if n > max_collect_rows:
            raise ValueError(
                f"Section '{section_name}' has {n:,} rows; refusing to collect to driver. "
                f"Check compute_kpis output (it should be a small KPI table)."
            )

        section_payload = {"metrics": {}, "breakdowns": {}}

        if use_collect_instead_of_pandas:
            rows = sdf_small.collect()
            for r in rows:
                kpi_name = str(r[kpi_col]).strip() if r[kpi_col] is not None else ""
                if not kpi_name:
                    continue

                current_val = _safe_float(r[current_col])
                baseline_val = _safe_float(r[baseline_col])

                pre_delta = _safe_float(r[delta_col]) if (delta_col in cols) else None

                delta_abs = None
                delta_pct = None
                if current_val is not None and baseline_val is not None:
                    delta_abs = current_val - baseline_val
                    if baseline_val != 0:
                        delta_pct = (current_val - baseline_val) / baseline_val

                delta_pct_precomputed = pre_delta if (prefer_precomputed_delta and pre_delta is not None) else None

                calls_delta = _safe_float(r[call_delta_col]) if (call_delta_col in cols) else None

                if _is_breakdown_kpi(kpi_name):
                    base, reason = _split_breakdown(kpi_name)
                    base_key = _to_snake(base)

                    section_payload["breakdowns"].setdefault(base_key, [])

                    # for breakdown rows, we often want "percentage points" change, not ratio
                    delta_pct_points = None
                    if current_val is not None and baseline_val is not None:
                        delta_pct_points = current_val - baseline_val

                    item = {
                        breakdown_reason_key: reason,
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_pct_points": delta_pct_points,
                        "kpi_label": kpi_name,
                    }

                    if calls_delta is not None:
                        item["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        item["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["breakdowns"][base_key].append(item)

                else:
                    metric_key = _to_snake(kpi_name)
                    metric_obj = {
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_abs": delta_abs,
                        "delta_pct": delta_pct,
                        "kpi_label": kpi_name,
                    }

                    if calls_delta is not None:
                        metric_obj["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        metric_obj["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["metrics"][metric_key] = metric_obj

        else:
            # fallback to pandas (kept for completeness)
            pdf = sdf_small.toPandas()
            for _, row in pdf.iterrows():
                kpi_name = str(row.get(kpi_col, "")).strip()
                if not kpi_name:
                    continue

                current_val = _safe_float(row.get(current_col))
                baseline_val = _safe_float(row.get(baseline_col))
                pre_delta = _safe_float(row.get(delta_col)) if (delta_col in cols) else None

                delta_abs = None
                delta_pct = None
                if current_val is not None and baseline_val is not None:
                    delta_abs = current_val - baseline_val
                    if baseline_val != 0:
                        delta_pct = (current_val - baseline_val) / baseline_val

                delta_pct_precomputed = pre_delta if (prefer_precomputed_delta and pre_delta is not None) else None
                calls_delta = _safe_float(row.get(call_delta_col)) if (call_delta_col in cols) else None

                if _is_breakdown_kpi(kpi_name):
                    base, reason = _split_breakdown(kpi_name)
                    base_key = _to_snake(base)
                    section_payload["breakdowns"].setdefault(base_key, [])

                    delta_pct_points = None
                    if current_val is not None and baseline_val is not None:
                        delta_pct_points = current_val - baseline_val

                    item = {
                        breakdown_reason_key: reason,
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_pct_points": delta_pct_points,
                        "kpi_label": kpi_name,
                    }

                    if calls_delta is not None:
                        item["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        item["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["breakdowns"][base_key].append(item)

                else:
                    metric_key = _to_snake(kpi_name)
                    metric_obj = {
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_abs": delta_abs,
                        "delta_pct": delta_pct,
                        "kpi_label": kpi_name,
                    }

                    if calls_delta is not None:
                        metric_obj["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        metric_obj["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["metrics"][metric_key] = metric_obj

        out[section_name] = section_payload

    return out


### JSON Filtering - Removing Unimportant Metrics Before API Call

In [0]:
import math
from typing import Any, Dict, List, Optional, Tuple

def filter_and_compact_kpi_json(
    kpi_json: Dict[str, Any],
    *,
    # -----------------------
    # Filtering thresholds
    # -----------------------
    # Drop whole sections with tiny volume (based on section gross calls)
    min_section_gross_calls: float = 0.0,

    # Metric-level volume gates:
    # Now that per-metric call counts are removed, we use:
    #   - metric["calls_delta"] if present? (not a volume)
    #   - otherwise section gross calls as the volume proxy
    min_metric_current_calls: float = 25.0,

    # Metric impact gates (keeps metric if ANY of these pass)
    min_abs_delta_pct: float = 0.03,          # e.g. 0.03 == 3% relative change
    min_abs_delta_abs: float = 0.0,           # optional for count KPIs; 0 disables if left at 0
    min_abs_delta_pp: float = 0.01,           # for rate-like metrics when delta_abs is “pp” in raw units (0.01==1pp)

    # Always keep these metric keys (snake keys in your JSON), regardless of thresholds
    always_keep_metrics: Tuple[str, ...] = (
        "gross_calls",
        "entered_compass_calls_count",
        "entered_compass_rate",
        "queue_calls",
        "queue_to_gross_rate",
        "abandonment_rate",
        "net_calls",
        "orders",
        "transactional_conversion_rate",
        "gross_conversion_rate",
        "total_revenue",
        "revenue_per_order",
        "revenue_per_net_call",
        "revenue_per_gross_call",
    ),

    # Breakdowns: keep top-N by absolute delta pct points (pp), after filtering
    breakdown_top_n: int = 8,
    min_breakdown_abs_delta_pp: float = 0.01,  # 1pp

    # Since per-breakdown current_calls may also be absent now, this is a best-effort gate:
    # - If item has current_calls, apply threshold
    # - Else, allow it through (or you can choose to gate by section gross)
    min_breakdown_current_calls: float = 25.0,

    # -----------------------
    # Compaction controls
    # -----------------------
    # Removed "current_calls" from defaults; add "calls_delta" instead
    keep_metric_fields: Tuple[str, ...] = ("current", "weekday_baseline_mean", "delta_abs", "delta_pct", "calls_delta"),
    keep_breakdown_fields: Tuple[str, ...] = ("reason", "current", "weekday_baseline_mean", "delta_pct_points", "calls_delta"),
    drop_filter_summary: bool = True,
    drop_empty_breakdowns: bool = True,
    drop_zero_volume_metrics: bool = True,
    round_digits: Optional[int] = 6,
) -> Dict[str, Any]:
    """
    Updated to match KPI JSON after removing per-step call counts:
      - No longer expects metric["current_calls"] / baseline calls.
      - Uses section gross calls as volume proxy for metric filtering.
      - Keeps "calls_delta" (from Call_Count_Delta) when present.

    Assumes input structure like:
      {
        "<section>": {
          "metrics": { "<metric_key>": { ... } },
          "breakdowns": { "<base_key>": [ { ... }, ... ] },
          "filter_summary": { ... }   # optional
        },
        ...
      }
    """

    def _is_nan(x: Any) -> bool:
        return isinstance(x, float) and math.isnan(x)

    def _clean(x: Any) -> Any:
        if x is None:
            return None
        if _is_nan(x):
            return None
        if isinstance(x, float) and math.isinf(x):
            return None
        return x

    def _round(v: Any) -> Any:
        v = _clean(v)
        if v is None:
            return None
        if round_digits is None:
            return v
        if isinstance(v, float):
            return round(v, round_digits)
        return v

    def _get_float(o: Dict[str, Any], key: str) -> Optional[float]:
        v = _clean(o.get(key))
        try:
            return float(v) if v is not None else None
        except Exception:
            return None

    def _metric_volume(metric_obj: Dict[str, Any], section_gross: Optional[float]) -> Optional[float]:
        # Per-metric call counts are gone; use section gross as best proxy.
        return section_gross

    def _metric_keep(metric_key: str, metric_obj: Dict[str, Any], section_gross: Optional[float]) -> bool:
        if metric_key in always_keep_metrics:
            return True

        cur = _get_float(metric_obj, "current")
        if cur is None:
            return False

        if drop_zero_volume_metrics:
            vol0 = _metric_volume(metric_obj, section_gross)
            if vol0 is not None and vol0 == 0:
                return False

        vol = _metric_volume(metric_obj, section_gross)
        if vol is not None and vol < min_metric_current_calls:
            return False

        # Impacts
        d_pct = _get_float(metric_obj, "delta_pct")
        d_abs = _get_float(metric_obj, "delta_abs")  # could be pp for rates or abs change for counts

        passes = False
        if d_pct is not None and abs(d_pct) >= min_abs_delta_pct:
            passes = True
        if (not passes) and (min_abs_delta_abs > 0) and (d_abs is not None) and abs(d_abs) >= min_abs_delta_abs:
            passes = True
        if (not passes) and (d_abs is not None) and abs(d_abs) >= min_abs_delta_pp:
            # For rates, delta_abs is typically pp in raw units (0.01 == 1pp)
            passes = True

        return passes

    def _compact_metric(metric_obj: Dict[str, Any]) -> Dict[str, Any]:
        o = metric_obj or {}
        outm = {}
        for f in keep_metric_fields:
            if f in o:
                outm[f] = _round(o.get(f))
        return {k: v for k, v in outm.items() if v is not None}

    def _compact_breakdown_item(item: Dict[str, Any]) -> Dict[str, Any]:
        it = item or {}
        outb = {}
        for f in keep_breakdown_fields:
            if f in it:
                outb[f] = _round(it.get(f))
        return {k: v for k, v in outb.items() if v is not None}

    filtered_compacted: Dict[str, Any] = {}

    for section_name, section in (kpi_json or {}).items():
        section = section or {}
        metrics_in = section.get("metrics", {}) or {}
        breakdowns_in = section.get("breakdowns", {}) or {}

        # Section gross calls (best-effort)
        section_gross = None
        if "gross_calls" in metrics_in:
            # with new JSON, gross_calls likely has only "current" (and maybe calls_delta)
            section_gross = _get_float(metrics_in["gross_calls"], "current")

        if section_gross is not None and section_gross < min_section_gross_calls:
            continue

        # ---- Metrics: filter + compact ----
        metrics_out: Dict[str, Any] = {}
        for metric_key, metric_obj in metrics_in.items():
            metric_obj = metric_obj or {}

            if not _metric_keep(metric_key, metric_obj, section_gross):
                continue

            cm = _compact_metric(metric_obj)
            if not cm:
                continue

            # Optional final drop: if section volume is 0, drop metric
            if drop_zero_volume_metrics and section_gross is not None and section_gross == 0:
                continue

            metrics_out[metric_key] = cm

        # If a section ends up empty, skip it
        if not metrics_out:
            continue

        # ---- Breakdowns: filter + rank + compact ----
        breakdowns_out: Dict[str, List[Dict[str, Any]]] = {}
        for base_key, items in breakdowns_in.items():
            kept: List[Tuple[float, Dict[str, Any]]] = []

            for item in (items or []):
                item = item or {}
                cur = _get_float(item, "current")
                if cur is None:
                    continue

                # volume gate (best-effort): only apply if current_calls exists
                bvol = _get_float(item, "current_calls")
                if bvol is not None and bvol < min_breakdown_current_calls:
                    continue
                if drop_zero_volume_metrics and bvol is not None and bvol == 0:
                    continue

                dpp = _get_float(item, "delta_pct_points")
                if dpp is None or abs(dpp) < min_breakdown_abs_delta_pp:
                    continue

                kept.append((abs(dpp), item))

            if not kept:
                continue

            kept.sort(key=lambda x: x[0], reverse=True)
            top_items = [it for _, it in kept[:breakdown_top_n]]

            compacted_items = []
            for it in top_items:
                cb = _compact_breakdown_item(it)
                if cb:
                    compacted_items.append(cb)

            if compacted_items:
                breakdowns_out[base_key] = compacted_items

        section_out: Dict[str, Any] = {"metrics": metrics_out}
        if not (drop_empty_breakdowns and not breakdowns_out):
            section_out["breakdowns"] = breakdowns_out

        if not drop_filter_summary and "filter_summary" in section:
            fs = section.get("filter_summary") or {}
            section_out["filter_summary"] = {k: _round(v) for k, v in fs.items() if _round(v) is not None}

        filtered_compacted[section_name] = section_out

    return filtered_compacted


# Report Generation From JSON

## First Pass

### Previous Reports

Just looping through the previous 7 days reports and summarizing them - almost executing level, but making sure to note small trends as well to spot continuity. 

In [0]:
from __future__ import annotations

import json
import re
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from docx import Document


_SUMMARY_RE = re.compile(r"^(?P<mm>\d{2})_(?P<dd>\d{2})_(?P<yyyy>\d{4})_summary\.docx$", re.IGNORECASE)


def _parse_summary_date_from_filename(filename: str) -> Optional[datetime]:
    m = _SUMMARY_RE.match(filename)
    if not m:
        return None
    mm, dd, yyyy = int(m["mm"]), int(m["dd"]), int(m["yyyy"])
    return datetime(yyyy, mm, dd)


def _read_docx_text(docx_path: Path) -> str:
    doc = Document(str(docx_path))
    parts: List[str] = []
    for p in doc.paragraphs:
        t = (p.text or "").strip()
        if t:
            parts.append(t)
    return "\n".join(parts)


def _extract_json_from_doc_text(doc_text: str) -> Dict[str, Any]:
    """
    Best-effort extraction of a JSON object from a docx text blob.
    Assumes the doc contains a JSON block OR at least one {...} object.
    """
    doc_text = doc_text.strip()

    # If the whole thing is JSON, great.
    try:
        obj = json.loads(doc_text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Otherwise, try to extract the first JSON object substring.
    start = doc_text.find("{")
    end = doc_text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in summary docx text.")
    candidate = doc_text[start : end + 1]
    obj = json.loads(candidate)
    if not isinstance(obj, dict):
        raise ValueError("Extracted JSON is not an object.")
    return obj


def get_weekly_context(
    knowledge_dir: str | Path = "knowledge",
    lookback_summaries: int = 7,
) -> Dict[str, Any]:
    """
    Reads the most recent N summary .docx files in knowledge/summaries named:
      MM_DD_YYYY_summary.docx

    Each file is expected to contain a JSON object keyed by date, e.g.:
      {
        "MM_DD_YYYY": {
          "day_of_week": "...",
          "executive_summary": "...",
          "callouts": [...]
        }
      }

    Returns a single merged artifact:
      {
        "weekly_context": {
          "MM_DD_YYYY": {...},
          "MM_DD_YYYY": {...},
          ...
        }
      }

    If duplicate date keys occur, the newest file (by filename date) wins.
    """
    knowledge_dir = Path(knowledge_dir)
    summaries_dir = knowledge_dir / "summaries"
    summaries_dir.mkdir(parents=True, exist_ok=True)

    # Gather dated summary files
    dated: List[Tuple[datetime, Path]] = []
    for p in summaries_dir.glob("*_summary.docx"):
        dt = _parse_summary_date_from_filename(p.name)
        if dt:
            dated.append((dt, p))

    if not dated:
        return {"weekly_context": {}}

    # Sort newest -> oldest; take last N
    dated.sort(key=lambda x: x[0], reverse=True)
    selected = dated[:lookback_summaries]

    merged: Dict[str, Any] = {}

    for dt, path in selected:
        doc_text = _read_docx_text(path)
        obj = _extract_json_from_doc_text(doc_text)

        # Expect dict keyed by date; merge into merged
        # Newer summaries override older ones for the same date key.
        for date_key, payload in obj.items():
            merged[date_key] = payload

    # Return in the shape you can pass directly to your report LLM call
    return {"weekly_context": merged}


### Getting Other Business Context

Basically will make a small OpenAI call just to get proper Vectore Store queries. The returned blocks here will be passed along to the actual query for context. 

This call has a small context block hard-coded so that the data can at least be interpreted.

Getting vector store queries from OpenAI call.

In [0]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Dict, Optional

from dotenv import load_dotenv
from openai import OpenAI


def _find_dotenv(start_path: Path) -> Optional[Path]:
    """Walk upward from start_path looking for a .env file."""
    for parent in [start_path] + list(start_path.parents):
        candidate = parent / ".env"
        if candidate.exists():
            return candidate
    return None


def _load_env(dotenv_path: str | Path | None = None) -> None:
    """Load environment variables from .env (explicitly)."""
    if dotenv_path is not None:
        p = Path(dotenv_path)
        if not p.exists():
            raise FileNotFoundError(f".env not found at: {p}")
        load_dotenv(p, override=True)
        return

    found = _find_dotenv(Path.cwd())
    if found:
        load_dotenv(found, override=True)


def generate_vector_store_queries(
    *,
    kpi_json: Dict[str, Any],
    weekly_context: Optional[Dict[str, Any]] = None,
    base_business_context: str,
    n_queries: int = 10,
    model: str = "gpt-5-mini",
    # NEW:
    dotenv_path: str | Path | None = None,
    vector_store_env_var: str = "OPENAI_VECTOR_STORE_ID",
) -> Dict[str, Any]:
    """
    Uses OpenAI to turn (1) daily KPI JSON + (2) weekly summaries + (3) minimal business context
    into a set of vector-store search queries for retrieving more context.

    Loads OPENAI_API_KEY from .env via load_dotenv (or uses existing env if already set).
    Also reads OPENAI_VECTOR_STORE_ID from env and returns it for downstream retrieval functions.

    Returns:
      {
        "vector_store_id": "vs_..." | null,
        "queries": [
          {"query": "...", "why": "...", "priority": "High|Medium|Low", "filters": {...}},
          ...
        ]
      }
    """
    # Ensure env vars are present (OPENAI_API_KEY / OPENAI_VECTOR_STORE_ID)
    _load_env(dotenv_path)

    client = OpenAI()  # will read OPENAI_API_KEY from env

    system_instructions = (
        "You are an analyst assistant that generates targeted semantic-search queries for a vector store. "
        "Your job is to propose queries that will retrieve the missing business context needed to explain "
        "today's performance in a daily Energy Voice report."
    )

    user_prompt = f"""
Use the following inputs to generate exactly {n_queries} vector-store search queries.

INPUTS
1) Minimal business context (shared vocabulary + directional interpretation):
{base_business_context}

2) Today's KPI JSON (the data we are explaining):
{json.dumps(kpi_json, ensure_ascii=False)}

3) Weekly context (prior 7 daily summaries; may be empty):
{json.dumps(weekly_context or {}, ensure_ascii=False)}

TASK
Generate EXACTLY {n_queries} search queries to retrieve additional context from our knowledge base.
The queries should focus on explaining the likely drivers of today's KPI changes and any notable trends.
Use Energy Voice vocabulary (Compass, Q2G, ERCOT match, abandon, CIContact, CICredit, RPO, RPGC, partner mix, routing).

QUERY GUIDELINES
- Each query must be short (4–12 words) and specific.
- Avoid numbers and dates unless essential.
- Prefer “drivers/causes/playbook/experiment/routing/partner mix/incidents” style phrasing.
- Diversity: cover funnel stages (marketing mix → Compass/IVR → queue → agent → revenue).
- If you suspect an explanation could be an incident or program change, include at least one query for that.

OPTIONAL FILTERS (include only if helpful)
- doc_type: one of ["glossary","playbook","experiment","incident","partner_update","ops_update","faq"]
- product_area: one of ["marketing","compass","queue","agent","revenue","routing","partner_mix"]
- recency_days: integer (e.g., 30, 60, 90) when freshness matters

OUTPUT
Return STRICT JSON with this shape:
{{
  "queries": [
    {{
      "query": "...",
      "why": "1 sentence rationale",
      "priority": "High|Medium|Low",
      "filters": {{"doc_type": "...", "product_area": "...", "recency_days": 60}}
    }}
  ]
}}
""".strip()

    schema = {
        "name": "vector_store_query_plan",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "queries": {
                    "type": "array",
                    "minItems": n_queries,
                    "maxItems": n_queries,
                    "items": {
                        "type": "object",
                        "properties": {
                            "query": {"type": "string"},
                            "why": {"type": "string"},
                            "priority": {"type": "string", "enum": ["High", "Medium", "Low"]},
                            "filters": {
                                "anyOf": [
                                    {
                                    "type": "object",
                                    "properties": {},
                                    "additionalProperties": False
                                    },
                                    {
                                    "type": "object",
                                    "properties": {
                                        "doc_type": {
                                        "type": ["string", "null"],
                                        "enum": [
                                            "glossary","playbook","experiment","incident",
                                            "partner_update","ops_update","faq", None
                                        ]
                                        },
                                        "product_area": {
                                        "type": ["string", "null"],
                                        "enum": [
                                            "marketing","compass","queue","agent",
                                            "revenue","routing","partner_mix", None
                                        ]
                                        },
                                        "recency_days": {"type": ["integer", "null"], "minimum": 1, "maximum": 3650}
                                    },
                                    "required": ["doc_type", "product_area", "recency_days"],
                                    "additionalProperties": False
                                    }
                                ]
                                }
                                ,
                        },
                        "required": ["query", "why", "priority", "filters"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["queries"],
            "additionalProperties": False,
        },
    }

    resp = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_instructions},
            {"role": "user", "content": user_prompt},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": schema["name"],
                "strict": True,
                "schema": schema["schema"],
            }
        },
    )

    raw = (resp.output_text or "").strip()
    try:
        plan = json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError(f"Model did not return JSON. Output was:\n{raw}")
        plan = json.loads(raw[start : end + 1])

    return {
        "vector_store_id": os.environ.get(vector_store_env_var),
        "queries": plan["queries"],
    }


Querying the vector store

In [0]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from openai import OpenAI

# reuse your existing _find_dotenv / _load_env if defined; otherwise simple loader:
def _find_dotenv(start_path: Path) -> Optional[Path]:
    for parent in [start_path] + list(start_path.parents):
        candidate = parent / ".env"
        if candidate.exists():
            return candidate
    return None

def _load_env(dotenv_path: str | Path | None = None) -> None:
    if dotenv_path is not None:
        p = Path(dotenv_path)
        if not p.exists():
            raise FileNotFoundError(f".env not found at: {p}")
        load_dotenv(p, override=True)
        return
    found = _find_dotenv(Path.cwd())
    if found:
        load_dotenv(found, override=True)

def _safe_get(obj: Any, attr: str, default=None):
    """Try dict access then attribute access."""
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(attr, default)
    return getattr(obj, attr, default)

def _item_to_dict(item: Any) -> Dict[str, Any]:
    """Convert SDK item to plain dict for introspection and access."""
    if item is None:
        return {}
    if isinstance(item, dict):
        return item
    # pydantic v2
    if hasattr(item, "model_dump"):
        try:
            return item.model_dump()
        except Exception:
            pass
    # pydantic v1
    if hasattr(item, "dict"):
        try:
            return item.dict()
        except Exception:
            pass
    # fallback
    if hasattr(item, "__dict__"):
        return {k: v for k, v in item.__dict__.items() if not k.startswith("_")}
    return {"repr": repr(item)}

def _extract_text_from_content(content: List[Any]) -> str:
    """
    Tolerant extractor: accepts dicts or objects, tries common field names and types.
    """
    parts: List[str] = []
    if not content:
        return ""
    for c in content:
        # Normalize to dict-ish view for easier checks
        cdict = _item_to_dict(c)
        # Common explicit 'text' segments
        ctype = cdict.get("type") or cdict.get("segment_type") or cdict.get("kind") or None
        ctext = cdict.get("text") or cdict.get("content") or cdict.get("body") or cdict.get("value") or None

        # If type is explicitly a text-like type, accept it
        if ctype and isinstance(ctype, str) and ctype.lower() in ("text", "plain_text", "paragraph", "content"):
            if isinstance(ctext, str) and ctext.strip():
                parts.append(ctext.strip())
                continue

        # If there was no explicit type, but there is a text-like field, accept it
        if isinstance(ctext, str) and ctext.strip():
            parts.append(ctext.strip())
            continue

        # Some SDK shapes present nested structures; attempt to stringify fallback
        # e.g., {"data": [{"text":"..."}]} or object fields
        # Try to find deepest string values heuristically
        if isinstance(cdict, dict):
            for candidate_key in ("text", "content", "body", "value", "excerpt"):
                cand = cdict.get(candidate_key)
                if isinstance(cand, str) and cand.strip():
                    parts.append(cand.strip())
                    break
            else:
                # Last resort: flatten small primitives in the dict
                flat_vals = []
                for v in cdict.values():
                    if isinstance(v, str) and v.strip():
                        flat_vals.append(v.strip())
                    elif isinstance(v, (int, float)):
                        flat_vals.append(str(v))
                if flat_vals:
                    parts.append(" ".join(flat_vals))
    return "\n".join(parts).strip()

def _build_vector_store_filters(file_attrs: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Same behavior as your previous builder; skip recency_days."""
    if not file_attrs:
        return None
    keep = {k: v for k, v in file_attrs.items() if v not in (None, "", [], {})}
    if not keep:
        return None
    comparisons: List[Dict[str, Any]] = []
    for k, v in keep.items():
        if k == "recency_days":
            continue
        comparisons.append({"type": "eq", "key": k, "value": v})
    if not comparisons:
        return None
    if len(comparisons) == 1:
        return comparisons[0]
    return {"type": "and", "filters": comparisons}

def retrieve_vector_store_blocks(
    *,
    vector_store_id: str | None = None,
    query_plan: Dict[str, Any],
    max_results_per_query: int = 5,
    score_threshold: float = 0.0,
    dedupe_on_text: bool = True,
    save_path: str | Path | None = None,
    dotenv_path: str | Path | None = None,
    vector_store_env_var: str = "OPENAI_VECTOR_STORE_ID",
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Robust retrieval: tries both attribute_filter and filters params and tolerates varied segment shapes.
    """
    _load_env(dotenv_path)

    if not vector_store_id:
        vector_store_id = os.environ.get(vector_store_env_var)
    if not vector_store_id:
        raise ValueError(
            "vector_store_id was not provided and no vector store id was found in env. "
            f"Set {vector_store_env_var}=vs_... in your .env or pass vector_store_id=..."
        )

    client = OpenAI()

    queries = query_plan.get("queries", [])
    if not isinstance(queries, list) or not queries:
        raise ValueError("query_plan must contain a non-empty 'queries' list.")

    blocks: Dict[str, Any] = {}
    seen_texts: set[str] = set()
    block_num = 1

    # Helper that tries several search parameter names and returns the result tuple (res, used_param)
    def _search_with_tryouts(qtext: str, attr_filter: Optional[Dict[str, Any]]):
        # 1) Try attribute_filter (newer SDK)
        try:
            res = client.vector_stores.search(
                vector_store_id=vector_store_id,
                query=qtext,
                max_num_results=max_results_per_query,
                ranking_options={"score_threshold": score_threshold},
            )
            # If it returned without throwing, return it (we'll check data length)
            return res, "attribute_filter"
        except Exception as e:
            if verbose:
                print(f"[search] attribute_filter raised: {type(e).__name__}: {e}")
        # 2) Try filters (older SDK)
        try:
            res = client.vector_stores.search(
                vector_store_id=vector_store_id,
                query=qtext,
                max_num_results=max_results_per_query,
                filters=attr_filter,
                ranking_options={"score_threshold": score_threshold},
            )
            return res, "filters"
        except Exception as e:
            if verbose:
                print(f"[search] filters raised: {type(e).__name__}: {e}")
        # 3) Last resort: try no filters at all
        try:
            res = client.vector_stores.search(
                vector_store_id=vector_store_id,
                query=qtext,
                max_num_results=max_results_per_query,
                ranking_options={"score_threshold": score_threshold},
            )
            return res, "no_filters"
        except Exception as e:
            # Give up for this query
            if verbose:
                print(f"[search] no-filters raised: {type(e).__name__}: {e}")
            return None, None

    for qi, q in enumerate(queries, start=1):
        query_text = (q.get("query") or "").strip()
        if not query_text:
            if verbose:
                print(f"[{qi}] skipping empty query object keys={list(q.keys())}")
            continue

        raw_filters = q.get("filters") or {}
        if not isinstance(raw_filters, dict):
            raw_filters = {}
        attribute_filter = _build_vector_store_filters(raw_filters)

        if verbose:
            print(f"[{qi}] searching for: {query_text!r} (filters={raw_filters})")

        res, used = _search_with_tryouts(query_text, attribute_filter)
        if res is None:
            if verbose:
                print(f"[{qi}] search attempts failed for query: {query_text!r}")
            continue

        # Normalize results to list
        res_data = getattr(res, "data", None) or _item_to_dict(res).get("data") or []
        nres = len(res_data or [])
        if verbose:
            print(f"[{qi}] search returned {nres} result(s) using param: {used}")

        if nres == 0:
            # Nothing returned for this query — continue to next query
            if verbose:
                print(f"[{qi}] no hits for query_text={query_text!r}")
            continue

        # Process each returned item
        for ri, item in enumerate(res_data, start=1):
            item_d = _item_to_dict(item)
            content = item_d.get("content") or []
            # Fallback: some shapes put text at item_d.get("text") directly
            if not content and isinstance(item_d.get("text"), str):
                content = [{"type": "text", "text": item_d.get("text")}]

            chunk_text = _extract_text_from_content(content)
            if not chunk_text:
                # If no chunk text, try to look for nested content keys
                # e.g., item_d.get("contents") or item_d.get("segments")
                for alt in ("contents", "segments", "data"):
                    altv = item_d.get(alt)
                    if altv:
                        chunk_text = _extract_text_from_content(altv)
                        if chunk_text:
                            break

            if not chunk_text:
                # nothing to add from this item
                if verbose:
                    print(f"[{qi}.{ri}] item had no extractable text; keys: {list(item_d.keys())}")
                continue

            if dedupe_on_text:
                norm = " ".join(chunk_text.split())
                if norm in seen_texts:
                    if verbose:
                        print(f"[{qi}.{ri}] duplicate text skipped")
                    continue
                seen_texts.add(norm)

            score = item_d.get("score")
            file_id = item_d.get("file_id")
            filename = item_d.get("filename")
            attributes = item_d.get("attributes") or {}

            blocks[str(block_num)] = {
                "block_id": block_num,
                "text": chunk_text,
                "score": score,
                "file_id": file_id,
                "filename": filename,
                "attributes": attributes or {},
                "retrieval": {
                    "query_index": qi,
                    "query": query_text,
                    "result_rank": ri,
                    "priority": q.get("priority"),
                    "why": q.get("why"),
                    "requested_filters": raw_filters,
                    "applied_filters": attribute_filter,
                    "search_param_used": used,
                },
            }
            if verbose:
                print(f"[{qi}.{ri}] added block #{block_num} (score={score}) filename={filename}")
            block_num += 1

    artifact = {"blocks": blocks}

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")

    return artifact


## Second Pass

Putting it all together here - this is where prompting gets pretty tough. Gonna start with executive summary, then do drivers, then potential analyses, then potentially figures/pacing numbers. 

### Executive Summary

In [0]:
from __future__ import annotations

import json
from typing import Any, Dict, List, Optional

from openai import OpenAI


def _top_block_texts(retrieved_blocks: Optional[Dict[str, Any]], max_blocks: int = 8) -> str:
    if not retrieved_blocks:
        return ""
    blocks = retrieved_blocks.get("blocks", {})
    try:
        ordered = sorted(blocks.values(), key=lambda b: int(b.get("block_id", 0)))
    except Exception:
        ordered = list(blocks.values())

    texts: List[str] = []
    for b in ordered[:max_blocks]:
        t = (b.get("text") or "").strip()
        if t:
            texts.append(t if len(t) <= 2000 else t[:2000] + " ...")
    return "\n\n".join(f"BLOCK[{i+1}]: {txt}" for i, txt in enumerate(texts))


def _extract_response_text(resp: Any) -> str:
    if resp is None:
        return ""

    t = getattr(resp, "output_text", None)
    if isinstance(t, str) and t.strip():
        return t.strip()

    out = getattr(resp, "output", None)
    if isinstance(out, list):
        chunks: List[str] = []
        for item in out:
            content = getattr(item, "content", None)
            if not isinstance(content, list):
                continue
            for part in content:
                ptype = getattr(part, "type", None) if not isinstance(part, dict) else part.get("type")
                if ptype in ("output_text", "text"):
                    ptext = getattr(part, "text", None) if not isinstance(part, dict) else part.get("text")
                    if ptext:
                        chunks.append(str(ptext))
        if chunks:
            return "\n".join(chunks).strip()

    try:
        return str(resp.output[0].content[0].text).strip()
    except Exception:
        pass

    if hasattr(resp, "model_dump"):
        try:
            d = resp.model_dump()
            if isinstance(d, dict) and isinstance(d.get("output_text"), str):
                return d["output_text"].strip()
        except Exception:
            pass

    return ""


def _get_tokenizer(model: str):
    """
    Best-effort tokenizer. Uses tiktoken if available; otherwise falls back to a rough heuristic.
    """
    try:
        import tiktoken  # type: ignore

        # tiktoken doesn't know every new model name; fall back safely.
        try:
            enc = tiktoken.encoding_for_model(model)
        except Exception:
            enc = tiktoken.get_encoding("o200k_base")
        return ("tiktoken", enc)
    except Exception:
        return ("heuristic", None)


def _count_tokens(text: str, tokenizer_kind: str, enc: Any) -> int:
    if not text:
        return 0
    if tokenizer_kind == "tiktoken" and enc is not None:
        return len(enc.encode(text))
    # heuristic: ~4 chars per token (very rough, but safe for truncation)
    return max(1, len(text) // 4)


def _truncate_to_tokens(text: str, max_tokens: int, tokenizer_kind: str, enc: Any) -> str:
    if max_tokens <= 0 or not text:
        return ""
    if _count_tokens(text, tokenizer_kind, enc) <= max_tokens:
        return text

    if tokenizer_kind == "tiktoken" and enc is not None:
        toks = enc.encode(text)
        toks = toks[:max_tokens]
        out = enc.decode(toks)
        return out + " ..."

    # heuristic truncation by chars
    approx_chars = max_tokens * 4
    return (text[:approx_chars] + " ...") if len(text) > approx_chars else text


def generate_executive_summary_for_report_text(
    *,
    kpi_json: Dict[str, Any],
    weekly_context: Optional[Dict[str, Any]],
    retrieved_blocks: Optional[Dict[str, Any]] = None,
    model: str = "gpt-5-mini",
    max_retrieved_blocks_in_prompt: int = 6,
    max_output_tokens: int = 2500,
    # NEW: provide a context window guess; used only for truncation. Override if you know the model's real window.
    model_context_window_tokens: int = 16000,
    # NEW: only allow prompt (input) up to 80% of the remaining budget after reserving output tokens.
    input_budget_fraction: float = 0.80,
    debug: bool = True,
) -> str:
    client = OpenAI()

    tokenizer_kind, enc = _get_tokenizer(model)

    user_instructions = """
You are an experienced operations analyst writing a senior-leadership executive summary
for Energy Voice.

Write a clear, narrative executive summary (2–3 short paragraphs) that walks through
yesterday's performance from top-of-funnel to revenue outcome.

IMPORTANT CONTEXT ABOUT THE INPUT DATA:
- If a metric or field is missing from the JSON, it was intentionally removed because it was insignificant
  for yesterday (not an error, not “not reported”). Do not call it out as missing unless it creates real ambiguity.
- Call volume often lifts at the beginning of the month because customers call about existing plans and setting up
  new service. Use this as light seasonal context when interpreting volume swings (don’t over-attribute).
- Bots never reach the queue. Automated or bot traffic is filtered out in the first Compass question and should be
  treated as top-of-funnel noise, not queue or agent-impacting volume.
- A “contact call” is a net call where all required contact information is completed.
- A “credit call” is a net call where credit has been successfully run.
- The net call funnel progresses in order: contact → credit → pitch → conversion. Use this sequence when describing
  funnel health and drop-off.
- The most important and nuances Compass step is the initial question. This is where the most breakage is naturally, but also where bots are filtered out. So, you much carefully interpret the bot rate and funnel drop-off.

Guidance:
- Start with a high-level assessment of the day (better / worse / mixed vs recent trend).
- Then move through the funnel in plain language:
  volume → Compass filtering → queue exposure → conversion → revenue quality.
- Use specific metrics (revenue, RPGC, RPO, gross calls, gross conversion, net conversion, Q2G)
  only where they materially support the story. Do NOT list metrics mechanically.
- Focus on *what changed* and *why it likely changed*, not just what the numbers were.

What to emphasize (flow, not labeled sections):
- First, comment on Compass health in a conversational way: stability/behavior, first question breakage, step-level collection, leading into call quality, and how Compass filtering shaped the day’s outcomes. Ex: Funnel may open at the top, but this may cause breakage further down due to more low intent callers. 
- Then transition naturally to agent health: abandon rate, net calls, call into contact, calls into credit, net conversion, and revenue per order. Revenue per net call is the north star for agent performance. 

What to avoid:
- Do NOT mention call routing, routing logic, or queue routing behavior.
- Do NOT call out specific partner changes, partner-level shifts, or partner performance.

Tone and framing:
- Write the way you would brief a VP or GM who wants signal, not a dashboard readout.
- Emphasize drivers and implications: Compass behavior, funnel friction, Q2G/abandon dynamics,
  agent effectiveness, marketing mix, or incidents — without operational minutiae.
- If drawing from retrieved context, qualify with language like
  “retrieved context suggests…” or “retrieved notes indicate…”.
- Avoid bullet points, tables, or step-level detail.

Style constraints:
- 2–3 short paragraphs, ~6–10 sentences total.
- Confident, analytical, and executive in tone.
- Use Energy Voice vocabulary naturally (Compass Breakage, Q2G, CIContact, CICredit, RPO, RPGC).

Return ONLY the executive summary text.
No JSON. No headings. No markdown.

    """.strip()

    # --- Build core (highest priority) sections first ---
    kpi_text = json.dumps(kpi_json or {}, ensure_ascii=False, indent=2)
    weekly_text = json.dumps(weekly_context or {}, ensure_ascii=False, indent=2)
    retrieved_snippet_full = _top_block_texts(retrieved_blocks, max_blocks=max_retrieved_blocks_in_prompt)

    # ... everything below remains exactly as in your original function ...

    # Compute input token budget:
    # total_context ~ model_context_window_tokens
    # reserve max_output_tokens for completion
    # allow prompt to use only input_budget_fraction of what's left (safety headroom)
    remaining_for_io = max(0, int(model_context_window_tokens) - int(max_output_tokens))
    input_budget = int(remaining_for_io * float(input_budget_fraction))

    # Always include instructions + KPI JSON if at all possible.
    base_parts = [
        "===== KPI JSON (yesterday) =====",
        kpi_text,
        "",
        "===== INSTRUCTIONS =====",
        user_instructions,
    ]
    base_prompt = "\n\n".join(base_parts)
    base_tokens = _count_tokens(base_prompt, tokenizer_kind, enc)

    # If even KPI+instructions exceed budget, truncate KPI JSON itself (last resort).
    if base_tokens > input_budget:
        # Keep instructions intact; truncate KPI block to fit.
        # Compute how many tokens remain for KPI JSON after accounting for the rest.
        instr_part = "\n\n".join(["===== INSTRUCTIONS =====", user_instructions])
        instr_tokens = _count_tokens(instr_part, tokenizer_kind, enc)

        header_part = "===== KPI JSON (yesterday) ====="
        header_tokens = _count_tokens(header_part, tokenizer_kind, enc)

        # Remaining tokens for KPI text (with some small glue newlines)
        glue_tokens = _count_tokens("\n\n\n\n", tokenizer_kind, enc)
        kpi_budget = max(0, input_budget - (instr_tokens + header_tokens + glue_tokens))

        kpi_text = _truncate_to_tokens(kpi_text, kpi_budget, tokenizer_kind, enc)
        base_parts = [
            "===== KPI JSON (yesterday) =====",
            kpi_text,
            "",
            "===== INSTRUCTIONS =====",
            user_instructions,
        ]
        base_prompt = "\n\n".join(base_parts)
        base_tokens = _count_tokens(base_prompt, tokenizer_kind, enc)

    # Now add optional context in priority order, truncating as needed:
    # 1) weekly_context (useful continuity)
    # 2) retrieved blocks (nice-to-have evidence)
    # We’ll still *truncate* retrieved first if we need to claw back tokens.
    weekly_block = "\n\n".join(
        [
            "===== WEEKLY CONTEXT (prior daily summaries JSON) =====",
            weekly_text,
            "",
        ]
    )
    retrieved_block = "\n\n".join(
        [
            "===== TOP RETRIEVED BLOCKS (selected) =====",
            retrieved_snippet_full,
            "",
        ]
    ) if retrieved_snippet_full else ""

    # Start with base + weekly + retrieved, then truncate context blocks to fit.
    parts = []
    # Keep original order you had: weekly, kpi, retrieved, instructions
    # But we already prepared base as KPI+instructions; rebuild final with same ordering.
    # We'll place KPI+instructions at the end of assembly to make budgeting easier.
    # (Ordering in the final prompt stays consistent with your original design.)

    # Determine how many tokens are available for context blocks combined.
    # We'll assemble: weekly + KPI + retrieved + instructions
    kpi_instr = base_prompt  # already KPI+instructions, possibly KPI truncated last-resort
    kpi_instr_tokens = _count_tokens(kpi_instr, tokenizer_kind, enc)

    context_budget = max(0, input_budget - kpi_instr_tokens)

    # Allocate context budget: try weekly first, then retrieved.
    weekly_tokens = _count_tokens(weekly_block, tokenizer_kind, enc)
    if weekly_tokens > context_budget:
        weekly_block = _truncate_to_tokens(weekly_block, context_budget, tokenizer_kind, enc)
        weekly_tokens = _count_tokens(weekly_block, tokenizer_kind, enc)

    remaining_after_weekly = max(0, context_budget - weekly_tokens)

    if retrieved_block:
        retrieved_tokens = _count_tokens(retrieved_block, tokenizer_kind, enc)
        if retrieved_tokens > remaining_after_weekly:
            retrieved_block = _truncate_to_tokens(retrieved_block, remaining_after_weekly, tokenizer_kind, enc)

    full_prompt = "\n\n".join(
        [b for b in [weekly_block.strip(), "===== KPI JSON (yesterday) =====\n" + kpi_text, retrieved_block.strip(), "===== INSTRUCTIONS =====\n" + user_instructions] if b]
    ).strip()

    # Final safety pass: ensure within budget (truncate retrieved first, then weekly).
    def _rebuild(weekly_b: str, retrieved_b: str) -> str:
        return "\n\n".join(
            [b for b in [weekly_b.strip(), "===== KPI JSON (yesterday) =====\n" + kpi_text, retrieved_b.strip(), "===== INSTRUCTIONS =====\n" + user_instructions] if b]
        ).strip()

    total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)
    if total_tokens > input_budget:
        # Reduce retrieved to 0 first if needed
        if retrieved_block:
            retrieved_block = _truncate_to_tokens(retrieved_block, 0, tokenizer_kind, enc)
            full_prompt = _rebuild(weekly_block, retrieved_block)
            total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)
        # Then reduce weekly if still needed
        if total_tokens > input_budget and weekly_block:
            # How many tokens left for weekly after fixed sections?
            fixed_prompt = _rebuild("", "")
            fixed_tokens = _count_tokens(fixed_prompt, tokenizer_kind, enc)
            weekly_allow = max(0, input_budget - fixed_tokens)
            weekly_block = _truncate_to_tokens(weekly_block, weekly_allow, tokenizer_kind, enc)
            full_prompt = _rebuild(weekly_block, "")
            total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if debug:
        print("DEBUG: tokenizer =", tokenizer_kind)
        print("DEBUG: model_context_window_tokens =", model_context_window_tokens)
        print("DEBUG: max_output_tokens =", max_output_tokens)
        print("DEBUG: input_budget_fraction =", input_budget_fraction)
        print("DEBUG: input_budget_tokens =", input_budget)
        print("DEBUG: final_prompt_tokens =", _count_tokens(full_prompt, tokenizer_kind, enc))
        print("DEBUG: weekly_context keys =", list((weekly_context or {}).keys())[:10])
        print("DEBUG: retrieved_snippet_full chars =", len(retrieved_snippet_full))
        if isinstance(kpi_json, dict):
            print("DEBUG: kpi_json top-level keys =", list(kpi_json.keys())[:20])

    resp = client.responses.create(
        model=model,
        input=full_prompt,
        max_output_tokens=max_output_tokens,
    )

    if debug:
        print("DEBUG: resp type =", type(resp))
        print("DEBUG: has output_text =", hasattr(resp, "output_text"))
        print("DEBUG: has output =", hasattr(resp, "output"))
        print("DEBUG: resp repr (truncated) =", repr(resp)[:800])

    text = _extract_response_text(resp)

    if debug and not text:
        try:
            out = getattr(resp, "output", None)
            print("DEBUG: resp.output (truncated) =", repr(out)[:1500])
        except Exception:
            pass

    return text


### Drivers/Deep-Dive

In [0]:
from __future__ import annotations

import json
from typing import Any, Dict, List, Optional

from openai import OpenAI


def _top_block_texts(retrieved_blocks: Optional[Dict[str, Any]], max_blocks: int = 8) -> str:
    """
    Return a compact concatenation of the top N retrieved blocks' text for inclusion in the prompt.
    Expects retrieved_blocks in the format returned by `retrieve_vector_store_blocks`:
      {"blocks": {"1": {...}, "2": {...}, ...}}
    """
    if not retrieved_blocks:
        return ""
    blocks = retrieved_blocks.get("blocks", {})
    try:
        ordered = sorted(blocks.values(), key=lambda b: int(b.get("block_id", 0)))
    except Exception:
        ordered = list(blocks.values())
    texts: List[str] = []
    for b in ordered[:max_blocks]:
        t = b.get("text", "").strip()
        if t:
            texts.append(t if len(t) <= 2000 else t[:2000] + " ...")
    return "\n\n".join(f"BLOCK[{i+1}]: {txt}" for i, txt in enumerate(texts))


# ---- Token helpers (best-effort) ----
def _get_tokenizer(model: str):
    """
    Best-effort tokenizer. Uses tiktoken if available; otherwise falls back to a rough heuristic.
    """
    try:
        import tiktoken  # type: ignore

        try:
            enc = tiktoken.encoding_for_model(model)
        except Exception:
            enc = tiktoken.get_encoding("o200k_base")
        return ("tiktoken", enc)
    except Exception:
        return ("heuristic", None)


def _count_tokens(text: str, tokenizer_kind: str, enc: Any) -> int:
    if not text:
        return 0
    if tokenizer_kind == "tiktoken" and enc is not None:
        return len(enc.encode(text))
    # heuristic: ~4 characters per token (rough)
    return max(1, len(text) // 4)


def _truncate_to_tokens(text: str, max_tokens: int, tokenizer_kind: str, enc: Any) -> str:
    """
    Truncate `text` to approximately `max_tokens` tokens.
    Returns an ellipsized string if truncated.
    """
    if max_tokens <= 0 or not text:
        return ""
    if _count_tokens(text, tokenizer_kind, enc) <= max_tokens:
        return text

    if tokenizer_kind == "tiktoken" and enc is not None:
        toks = enc.encode(text)
        toks = toks[:max_tokens]
        out = enc.decode(toks)
        return out + " ..."
    # fallback: char-based truncation
    approx_chars = max_tokens * 4
    return (text[:approx_chars] + " ...") if len(text) > approx_chars else text


def generate_step_level_drivers(
    *,
    kpi_json: Dict[str, Any],
    weekly_context: Optional[Dict[str, Any]],
    retrieved_blocks: Optional[Dict[str, Any]] = None,
    model: str = "gpt-5-mini",
    min_drivers: int = 3,
    max_retrieved_blocks_in_prompt: int = 6,
    # New/adjusted defaults for larger responses
    max_output_tokens: int = 4000,
    # Model context window guess (override if you know it)
    model_context_window_tokens: int = 16000,
    # Fraction of remaining context to actually use (safety headroom)
    input_budget_fraction: float = 0.80,
) -> Dict[str, Any]:
    """
    Same behavior as before but enforces a context token budget and allows a larger output token budget.
    Priority for preservation:
      1) Instructions & required output schema
      2) KPI JSON (yesterday)
      3) Weekly context (prior summaries)
      4) Retrieved evidence blocks
    Retrieved evidence is truncated/dropped first to preserve KPI + instructions.
    """
    client = OpenAI()

    tokenizer_kind, enc = _get_tokenizer(model)

    # Keep the business context short (but keep it present). Use compact JSON for weekly/kpi to save tokens.

    business_context = (
        "Energy Voice — Operating Notes (keep practical)\n"
        "- Funnel framing: volume (calls) -> Compass filtering -> queue rate -> net call -> conversion -> revenue quality.\n"
        "- Bots never reach the queue; automated/bot traffic is filtered out at the first Compass question.\n"
        "- CIContact and CICredit are sequential funnel steps: contact -> credit -> pitch -> conversion.\n"
        "  * Contact call = a net call with all required contact information completed.\n"
        "  * Credit call  = a net call with credit successfully run.\n"
        "- Start-of-month often has higher call volume because customers handle existing plans or set up new service.\n"
        "- Keep this operational; avoid generic business-speak.\n"
        "- Do NOT mention call routing or routing logic.\n"
        "- Do NOT mention specific partner changes.\n"
    )

    # Use compact JSON to reduce token counts (no indentation)
    kpi_text = json.dumps(kpi_json or {}, ensure_ascii=False, separators=(",", ":"))
    weekly_text = json.dumps(weekly_context or {}, ensure_ascii=False, separators=(",", ":"))
    retrieved_snippet_full = _top_block_texts(retrieved_blocks, max_blocks=max_retrieved_blocks_in_prompt)

    # Instruction + strict-output schema (must be kept)
    instructions_and_schema = """
TASK:
Identify at least {min_drivers} main drivers (step-level or volume-source metrics) that explain yesterday's performance.
Each driver should describe what changed, what it likely drove downstream, and what analyses to run next to understand it.

IMPORTANT CONTEXT ABOUT THE INPUT DATA:
- If a metric or field is missing from the JSON, assume it was intentionally removed because it was insignificant
  for yesterday (not an error, not “not reported”). Do not call it out as missing unless it creates real ambiguity.
  HOWEVER, explicitly note missing data when it would materially improve confidence in diagnosing funnel drivers.
- Call volume often lifts at the beginning of the month because customers call about existing plans and setting up
  new service. Use this as light seasonal context when interpreting volume swings (don’t over-attribute).
- Bots never reach the queue. Automated or bot traffic is filtered out in the first Compass question and should be
  treated as top-of-funnel noise, not queue or agent-impacting volume.
- A “contact call” is a net call where all required contact information is completed.
- A “credit call” is a net call where credit has been successfully run.
- The net call funnel progresses in order: contact → credit → pitch → conversion.
- The most important and nuanced Compass step is the initial question. This is where the most natural breakage occurs
  and where bots are filtered out. Interpret bot rate and early funnel drop-off carefully, especially when volume expands.

DEPTH & ANALYTICAL FRAME:
- Think in funnel flow, not isolated metrics:
  volume → Compass filtering / first-question breakage → queue exposure → conversion → revenue quality.
- Funnel expansion at the top can improve volume while introducing lower intent callers, which may surface as
  downstream breakage in contact, credit, or conversion. Call this out where relevant.
- Use Energy Voice vocabulary naturally (Compass Breakage, Q2G, CIContact, CICredit, RPGC, RPO).
- Stay operational and analytical; avoid generic business commentary or macro explanations.

WHAT TO EMPHASIZE ACROSS DRIVERS:
- Prioritize Compass health first: stability, first-question behavior, step-level collection (CIContact, CICredit),
  intent quality, and how Compass filtering shaped downstream outcomes.
- Then transition naturally to agent health: abandon rate, net calls, calls into contact, calls into credit,
  net conversion, and revenue per order.
- Revenue per net call is the north star for agent effectiveness—use it as the anchor when discussing agent impact.

WHAT TO AVOID:
- Do NOT mention call routing, routing logic, or queue routing behavior.
- Do NOT call out specific partner changes, partner-level shifts, or partner performance.

DRIVER CONTENT REQUIREMENTS:
For each driver return these fields:
- "title": one-line title (specific, operational)
- "what_happened": 1–2 short sentences describing the observed change at the step or volume-source level
- "what_it_drove": 1 short sentence describing the downstream KPI impact (conversion, revenue, RPGC, RPO, Q2G, etc.)
- "next_steps": an array of 3 short, analytical follow-ups to validate or deepen understanding
- "evidence": short note indicating supporting retrieved context OR helpful missing data

EVIDENCE FIELD GUIDANCE:
Use the "evidence" field to capture signal quality:
- Retrieved support (e.g., “retrieved notes indicate Compass instability in first question”)
- Helpful missing data (e.g., “missing: CIContact drop-off by first-question response would confirm breakage source”)
- Or "none" if not applicable

NEXT_STEPS GUIDANCE (ANALYSIS-FOCUSED):
Your "next_steps" should be investigative analyses such as:
- Segmenting volume and breakage by Compass first-question response or intent bucket.
- Comparing CIContact → CICredit progression rates before vs after volume shifts.
- Breaking out net calls and conversion by intent or call quality segment.
- Splitting marketing mix (site vs SERP, campaign, provider) to identify where low- or high-intent volume entered.
- Running quick validation checks before deep dives; keep all steps analytical.

OUTPUT:
Return STRICT JSON only with this exact top-level shape:
{
  "drivers": [
    {
      "title": "<string>",
      "what_happened": "<string>",
      "what_it_drove": "<string>",
      "next_steps": ["<string>", "<string>", "<string>"],
      "evidence": "<string>"
    }...
  ]
}
- Return only JSON, no explanatory text outside the JSON.
""".strip().format(min_drivers=min_drivers)

    # Compute token budgets
    remaining_for_io = max(0, int(model_context_window_tokens) - int(max_output_tokens))
    input_budget = int(remaining_for_io * float(input_budget_fraction))

    # Build the high-priority base prompt (instructions + KPI)
    base_parts = [
        "BUSINESS_CONTEXT:",
        business_context,
        "",
        "YDAY_KPI_JSON:",
        kpi_text,
        "",
        "INSTRUCTIONS_AND_SCHEMA:",
        instructions_and_schema,
        "",
    ]
    base_prompt = "\n".join(base_parts)
    base_tokens = _count_tokens(base_prompt, tokenizer_kind, enc)

    # If base exceeds budget (unlikely unless model window small), truncate KPI JSON last-resort
    if base_tokens > input_budget:
        # preserve instructions/schema; truncate KPI JSON proportionally
        instr_part = "\n".join(["INSTRUCTIONS_AND_SCHEMA:", instructions_and_schema])
        instr_tokens = _count_tokens(instr_part, tokenizer_kind, enc)
        header_tokens = _count_tokens("YDAY_KPI_JSON:", tokenizer_kind, enc)
        glue_tokens = _count_tokens("\n\n", tokenizer_kind, enc)
        kpi_budget = max(
            0,
            input_budget
            - (
                instr_tokens
                + header_tokens
                + glue_tokens
                + _count_tokens("BUSINESS_CONTEXT:", tokenizer_kind, enc)
            ),
        )
        kpi_text = _truncate_to_tokens(kpi_text, kpi_budget, tokenizer_kind, enc)
        base_parts = [
            "BUSINESS_CONTEXT:",
            business_context,
            "",
            "YDAY_KPI_JSON:",
            kpi_text,
            "",
            "INSTRUCTIONS_AND_SCHEMA:",
            instructions_and_schema,
            "",
        ]
        base_prompt = "\n".join(base_parts)
        base_tokens = _count_tokens(base_prompt, tokenizer_kind, enc)

    # Now prepare optional context blocks (weekly, retrieved)
    weekly_block = "\n".join(["WEEKLY_CONTEXT:", weekly_text, ""])
    retrieved_block = "\n".join(["RETRIEVED_EVIDENCE:", retrieved_snippet_full, ""]) if retrieved_snippet_full else ""

    # How many tokens left for optional context
    context_budget = max(0, input_budget - base_tokens)

    # First try to fit weekly_context (higher priority than retrieved evidence)
    weekly_tokens = _count_tokens(weekly_block, tokenizer_kind, enc)
    if weekly_tokens > context_budget:
        weekly_block = _truncate_to_tokens(weekly_block, context_budget, tokenizer_kind, enc)
        weekly_tokens = _count_tokens(weekly_block, tokenizer_kind, enc)

    remaining_after_weekly = max(0, context_budget - weekly_tokens)

    # Then allocate to retrieved evidence (truncate/dropping will happen here first)
    if retrieved_block:
        retrieved_tokens = _count_tokens(retrieved_block, tokenizer_kind, enc)
        if retrieved_tokens > remaining_after_weekly:
            retrieved_block = _truncate_to_tokens(retrieved_block, remaining_after_weekly, tokenizer_kind, enc)

    # Assemble final prompt in consistent order
    final_parts = [
        "BUSINESS_CONTEXT:",
        business_context,
        "",
        "WEEKLY_CONTEXT:",
        weekly_block.strip() if weekly_block else "{}",
        "",
        "YDAY_KPI_JSON:",
        kpi_text,
        "",
        "RETRIEVED_EVIDENCE:",
        retrieved_block.strip() if retrieved_block else "[none]",
        "",
        "INSTRUCTIONS_AND_SCHEMA:",
        instructions_and_schema,
    ]
    full_prompt = "\n".join([p for p in final_parts if p is not None])

    # Final safety pass: if we still exceed, drop/reduce retrieved, then weekly, then KPI
    total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)
    if total_tokens > input_budget:
        # drop retrieved entirely
        if retrieved_block:
            retrieved_block = ""
            full_prompt = full_prompt.replace(
                "\nRETRIEVED_EVIDENCE:\n" + retrieved_block.strip() + "\n",
                "\nRETRIEVED_EVIDENCE:\n[none]\n",
            )
            total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        # reduce weekly to fit
        fixed_prompt_without_weekly = "\n".join([p for p in final_parts if "WEEKLY_CONTEXT:" not in p and "BUSINESS_CONTEXT:" not in p])
        fixed_tokens = _count_tokens(fixed_prompt_without_weekly, tokenizer_kind, enc)
        weekly_allow = max(
            0,
            input_budget - fixed_tokens - _count_tokens("BUSINESS_CONTEXT:\n" + business_context + "\n", tokenizer_kind, enc),
        )
        weekly_block = _truncate_to_tokens(weekly_block, weekly_allow, tokenizer_kind, enc)
        # rebuild
        full_prompt = "\n".join(
            [
                "BUSINESS_CONTEXT:",
                business_context,
                "",
                "WEEKLY_CONTEXT:",
                weekly_block.strip() if weekly_block else "{}",
                "",
                "YDAY_KPI_JSON:",
                kpi_text,
                "",
                "RETRIEVED_EVIDENCE:",
                "[none]",
                "",
                "INSTRUCTIONS_AND_SCHEMA:",
                instructions_and_schema,
            ]
        )
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        # last resort: truncate KPI a bit to fit
        fixed_prompt = "\n".join(
            [
                "BUSINESS_CONTEXT:",
                business_context,
                "",
                "WEEKLY_CONTEXT:",
                weekly_block.strip() if weekly_block else "{}",
                "",
                "RETRIEVED_EVIDENCE:",
                "[none]",
                "",
                "INSTRUCTIONS_AND_SCHEMA:",
                instructions_and_schema,
            ]
        )
        fixed_tokens = _count_tokens(fixed_prompt, tokenizer_kind, enc)
        kpi_allow = max(0, input_budget - fixed_tokens)
        kpi_text = _truncate_to_tokens(kpi_text, kpi_allow, tokenizer_kind, enc)
        full_prompt = "\n".join(
            [
                "BUSINESS_CONTEXT:",
                business_context,
                "",
                "WEEKLY_CONTEXT:",
                weekly_block.strip() if weekly_block else "{}",
                "",
                "YDAY_KPI_JSON:",
                kpi_text,
                "",
                "RETRIEVED_EVIDENCE:",
                "[none]",
                "",
                "INSTRUCTIONS_AND_SCHEMA:",
                instructions_and_schema,
            ]
        )
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    # Execute the request with a larger output budget
    resp = client.responses.create(
        model=model,
        input=full_prompt,
        max_output_tokens=max_output_tokens,
    )

    raw = (getattr(resp, "output_text", None) or "").strip()

    # Defensive parsing: extract first JSON object substring if needed
    parsed: Dict[str, Any]
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end = raw.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError(f"Model did not return valid JSON. Output:\n{raw}")
        parsed = json.loads(raw[start : end + 1])

    # Validate/normalize parsed structure
    drivers = parsed.get("drivers")
    if not isinstance(drivers, list):
        raise ValueError("Parsed JSON missing 'drivers' list.")

    normalized_drivers: List[Dict[str, Any]] = []
    for d in drivers:
        if not isinstance(d, dict):
            continue
        title = str(d.get("title", "")).strip()
        what = str(d.get("what_happened", "")).strip()
        what_it_drove = str(d.get("what_it_drove", "")).strip()
        next_steps = d.get("next_steps") or []
        if not isinstance(next_steps, list):
            next_steps = [str(next_steps)]
        next_steps = [str(x).strip() for x in next_steps if str(x).strip()]
        while len(next_steps) < 3:
            next_steps.append("Analyze splits by marketing code IDs and site vs SERP to localize the change.")
        evidence = str(d.get("evidence", "")).strip() or "none"

        normalized_drivers.append(
            {
                "title": title or "Untitled driver",
                "what_happened": what or "No short description provided.",
                "what_it_drove": what_it_drove or "Downstream impact not specified.",
                "next_steps": next_steps[:5],
                "evidence": evidence,
            }
        )

    while len(normalized_drivers) < min_drivers:
        normalized_drivers.append(
            {
                "title": "Additional investigation needed",
                "what_happened": "Model did not provide enough drivers; further retrieval required.",
                "what_it_drove": "Unknown",
                "next_steps": [
                    "Analyze volume and conversion deltas by specific marketing code IDs to identify the source of change",
                    "Split marketing mix deeper (site vs SERP, campaign, brand/provider) and compare step performance",
                    "Segment callers by prior intent/behavior (e.g., moving vs switching) and re-check Q2G and conversion differences",
                ],
                "evidence": "none",
            }
        )

    return {"drivers": normalized_drivers, "output": raw}



### Potential Analyses/Things to Watch/Large Fluctuations

In [0]:
from __future__ import annotations

import re
import json
import ast
from typing import Any, Dict, List, Optional

from openai import OpenAI


def _top_block_texts(retrieved_blocks: Optional[Dict[str, Any]], max_blocks: int = 6) -> str:
    """
    Return a compact concatenation of the top N retrieved blocks' text for inclusion in the prompt.
    Expects retrieved_blocks in the format returned by `retrieve_vector_store_blocks`:
      {"blocks": {"1": {...}, "2": {...}, ...}}
    """
    if not retrieved_blocks:
        return ""
    blocks = retrieved_blocks.get("blocks", {})
    try:
        ordered = sorted(blocks.values(), key=lambda b: int(b.get("block_id", 0)))
    except Exception:
        ordered = list(blocks.values())
    texts: List[str] = []
    for b in ordered[:max_blocks]:
        t = b.get("text", "").strip()
        if t:
            texts.append(t if len(t) <= 2000 else t[:2000] + " ...")
    return "\n\n".join(f"BLOCK[{i+1}]: {txt}" for i, txt in enumerate(texts))


# --- Token helpers (best effort) ---
def _get_tokenizer(model: str):
    try:
        import tiktoken  # type: ignore

        try:
            enc = tiktoken.encoding_for_model(model)
        except Exception:
            enc = tiktoken.get_encoding("o200k_base")
        return ("tiktoken", enc)
    except Exception:
        return ("heuristic", None)


def _count_tokens(text: str, tokenizer_kind: str, enc: Any) -> int:
    if not text:
        return 0
    if tokenizer_kind == "tiktoken" and enc is not None:
        return len(enc.encode(text))
    # rough heuristic: ~4 chars per token
    return max(1, len(text) // 4)


def _truncate_to_tokens(text: str, max_tokens: int, tokenizer_kind: str, enc: Any) -> str:
    if max_tokens <= 0 or not text:
        return ""
    if _count_tokens(text, tokenizer_kind, enc) <= max_tokens:
        return text
    if tokenizer_kind == "tiktoken" and enc is not None:
        toks = enc.encode(text)
        toks = toks[:max_tokens]
        out = enc.decode(toks)
        return out + " ..."
    approx_chars = max_tokens * 4
    return (text[:approx_chars] + " ...") if len(text) > approx_chars else text


# --- Robust JSON sanitizer / loader ---
def sanitize_and_load_json(raw: str):
    """
    Try multiple strategies to parse 'raw' into JSON:
    Returns (parsed_object, cleaned_raw_used_for_parsing)
    Raises ValueError if not recoverable.
    """
    def try_json(s: str):
        try:
            return json.loads(s)
        except Exception:
            return None

    # 0) quick: direct parse
    parsed = try_json(raw)
    if parsed is not None:
        return parsed, raw

    # 1) extract first {...} .. last } (best-effort)
    first = raw.find("{")
    last = raw.rfind("}")
    if first != -1 and last != -1 and last > first:
        candidate = raw[first : last + 1]
        parsed = try_json(candidate)
        if parsed is not None:
            return parsed, candidate

    s = raw

    # 2) remove control characters that often break JSON display
    s = "".join(ch for ch in s if (31 < ord(ch) < 127) or ch in "\n\r\t")

    # 3) normalize smart quotes to straight quotes
    s = s.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")

    # 4) trim to first {...}
    first = s.find("{")
    last = s.rfind("}")
    if first != -1 and last != -1 and last > first:
        s = s[first : last + 1]

    # 5) remove trailing commas before } or ]
    s = re.sub(r",\s*(\}|])", r"\1", s)

    # 6) convert Python-like booleans/None to JSON
    s = re.sub(r"\bNone\b", "null", s)
    s = re.sub(r"\bTrue\b", "true", s)
    s = re.sub(r"\bFalse\b", "false", s)

    # 7) attempt to quote unquoted keys (heuristic)
    s = re.sub(r'([{,]\s*)([A-Za-z0-9_\- ]+)\s*:', r'\1"\2":', s)

    # 8) attempt to convert single-quoted strings to double-quoted (best-effort)
    s = re.sub(r"(?P<pre>[:\[,]\s*)'(?P<body>[^']*?)'(?P<post>\s*[,\]}])", r'\1"\2"\3', s)
    s = s.replace("':", '":').replace(",'", ',"')

    # try parse now
    parsed = try_json(s)
    if parsed is not None:
        return parsed, s

    # 9) Try to parse multiple {...} fragments and pick the largest parseable one
    raw_blocks = re.findall(r"\{(?:[^{}]|\{[^{}]*\})*?\}", s, flags=re.DOTALL)
    best = None
    for b in raw_blocks:
        p = try_json(b)
        if p is not None:
            if best is None or len(b) > len(best[1]):
                best = (p, b)
    if best is not None:
        return best[0], best[1]

    # 10) Last resort: try ast.literal_eval after swapping true/false/null to Python equivalents
    alt = s.replace("true", "True").replace("false", "False").replace("null", "None")
    try:
        p = ast.literal_eval(alt)
        return p, alt
    except Exception:
        pass

    preview = raw[:2000].replace("\n", "\\n")
    raise ValueError(
        "Failed to parse JSON from model output. Raw model output (first 2000 chars):\n"
        f"{preview}\n\n"
        "Consider: instructing the model to return compact JSON only (no surrounding text), "
        "or returning a small fallback JSON like {\"error\":\"...\"} when it cannot produce valid JSON."
    )

def generate_out_of_scope_analyses_and_alarms(
    *,
    kpi_json: Dict[str, Any],
    weekly_context: Optional[Dict[str, Any]],
    retrieved_blocks: Optional[Dict[str, Any]] = None,
    model: str = "gpt-5-mini",
    min_alerts: int = 3,
    min_monitor_items: int = 4,
    max_retrieved_blocks_in_prompt: int = 6,
    max_output_tokens: int = 2000,
    model_context_window_tokens: int = 16000,
    input_budget_fraction: float = 0.80,
    debug: bool = False,
) -> Dict[str, Any]:
    """
    Minimal version: build prompt as before, call the model, and return ONLY the raw model output
    as a string under "raw_model_output". No parsing or normalization is performed here.
    """
    client = OpenAI()
    tokenizer_kind, enc = _get_tokenizer(model)

    # compact JSON to reduce token footprint
    kpi_text = json.dumps(kpi_json or {}, ensure_ascii=False, separators=(",", ":"))
    weekly_text = json.dumps(weekly_context or {}, ensure_ascii=False, separators=(",", ":"))
    retrieved_snippet_full = _top_block_texts(retrieved_blocks, max_blocks=max_retrieved_blocks_in_prompt)

    # UPDATED: keep it practical + include routing + month-start context + avoid business-speak
    business_context = (
        "Energy Voice operating context, for sense-making rather than attribution. The funnel should be read as "
        "volume entering Compass, filtered heavily by the first question, then progressing through contact, credit, "
        "pitch, and finally conversion and revenue quality (RPGC/RPO). Bots never reach the queue and are filtered out "
        "in the first Compass question, so early breakage is both expected and meaningful when it changes materially. "
        "Start-of-month volume is often higher as customers deal with existing plans or set up new service; use that as "
        "light seasonal context only. Keep thinking operational and diagnostic, not explanatory or strategic. "
        "Do not reference call routing or specific partner changes."
    )

    # UPDATED: instructions emphasize 'out-of-scope' unknowns + analyses to get more context

    schema_and_instructions = f"""
    TASK & FORMAT (STRICT JSON):
    Return a JSON object with exactly three keys: "alerts", "monitoring_items", and "recommended_analyses".

    HOW TO THINK ABOUT THE DATA:
    Approach this as a diagnostic layer for ambiguity. These are not confirmed drivers, but signals that feel unusual,
    incomplete, or directionally important enough that leadership should know we are watching them or investigating further.

    Read performance through the full funnel in order: volume entering Compass, filtering and breakage at the first question,
    progression into contact and credit, then conversion and revenue quality. Remember that bots are filtered out immediately
    and never reach the queue, so shifts in early Compass behavior deserve careful interpretation rather than alarm by default.

    If a metric or field is missing from the JSON, assume it was intentionally removed because it was insignificant for yesterday.
    Do not treat omissions as errors. That said, if missing information meaningfully limits confidence in interpreting a signal,
    call that out explicitly and frame it as “this would help us understand what’s happening.”

    Use start-of-month seasonality as light context for volume only. Do not over-attribute.

    DEPTH & TONE:
    Write the way an experienced analyst would talk through unknowns with a GM or VP. Keep it practical, measured,
    and grounded in how the system actually behaves. Avoid generic business language or confident causal claims.
    Use Energy Voice vocabulary naturally (Compass breakage, CIContact, CICredit, Q2G, RPGC, RPO, SERP vs Site).

    WHAT THIS MODULE IS FOR:
    This is where we surface things that look off-pattern, fragile, or under-explained — not where we force a clean story.
    Prefer “this stands out and here’s how we should pressure-test it” over “this clearly caused X.”

    STRUCTURE REQUIREMENTS:

    1) alerts (at least {min_alerts}):
    These should read like short diagnostic callouts. Focus on sharp or surprising movements that don’t yet have a clear
    explanation — for example, sudden changes in Compass breakage, CIContact or CICredit progression, abandon or Q2G behavior,
    or revenue efficiency that doesn’t line up with volume or conversion. Each alert should point to the part of the funnel
    that likely needs attention (Compass, early funnel quality, agent execution, marketing mix), without asserting causality.

    2) monitoring_items (at least {min_monitor_items}):
    These are things that may not be broken yet, but feel unstable or directionally important enough to keep watching over
    the next few days. Think slow-burn risks, recurring volatility, early signs of funnel friction, or patterns that could
    compound if they persist.

    3) recommended_analyses (3–6 items):
    These should be framed as investigative questions or cuts of the data that would help explain the alerts or monitoring
    items. Keep them analytical and concrete. Examples include breaking volume or breakage by Compass first-question response,
    comparing contact-to-credit progression before and after volume shifts, splitting performance by SERP vs site or by
    specific marketing code IDs, or segmenting callers by intent archetype. Phrase these as “analyze X to understand Y,”
    not as fixes or actions.

    EVIDENCE HANDLING:
    If retrieved context supports an alert or concern, say so briefly (e.g., “retrieved notes suggest Compass instability”).
    If not, keep the language hypothesis-light. If clarity is limited by missing data, say what would help.

    OUTPUT RULES:
    - Return JSON only, no explanatory text outside the JSON.
    - Each list item should be written as a short paragraph (1–2 sentences), not a fragment or bullet.
    """.strip()

    # Compute token budget
    remaining_for_io = max(0, int(model_context_window_tokens) - int(max_output_tokens))
    input_budget = int(remaining_for_io * float(input_budget_fraction))

    # Build base prompt (keep compact)
    base_parts = [
        "BUSINESS_CONTEXT:",
        business_context,
        "",
        "YDAY_KPI_JSON:",
        kpi_text,
        "",
        "INSTRUCTIONS_AND_SCHEMA:",
        schema_and_instructions,
        "",
    ]
    base_prompt = "\n".join(base_parts)
    base_tokens = _count_tokens(base_prompt, tokenizer_kind, enc)

    # Truncate KPI if necessary (last-resort)
    if base_tokens > input_budget:
        instr_part = "\n".join(["INSTRUCTIONS_AND_SCHEMA:", schema_and_instructions])
        instr_tokens = _count_tokens(instr_part, tokenizer_kind, enc)
        header_tokens = _count_tokens("YDAY_KPI_JSON:", tokenizer_kind, enc)
        glue_tokens = _count_tokens("\n\n", tokenizer_kind, enc)
        business_tokens = _count_tokens("BUSINESS_CONTEXT:" + business_context, tokenizer_kind, enc)
        available_for_kpi = max(0, input_budget - (instr_tokens + header_tokens + glue_tokens + business_tokens))
        kpi_text = _truncate_to_tokens(kpi_text, available_for_kpi, tokenizer_kind, enc)
        base_parts = [
            "BUSINESS_CONTEXT:",
            business_context,
            "",
            "YDAY_KPI_JSON:",
            kpi_text,
            "",
            "INSTRUCTIONS_AND_SCHEMA:",
            schema_and_instructions,
            "",
        ]
        base_prompt = "\n".join(base_parts)
        base_tokens = _count_tokens(base_prompt, tokenizer_kind, enc)

    weekly_block = "\n".join(["WEEKLY_CONTEXT:", weekly_text, ""])
    retrieved_block = "\n".join(["RETRIEVED_EVIDENCE:", retrieved_snippet_full, ""]) if retrieved_snippet_full else ""

    context_budget = max(0, input_budget - base_tokens)

    weekly_tokens = _count_tokens(weekly_block, tokenizer_kind, enc)
    if weekly_tokens > context_budget:
        weekly_block = _truncate_to_tokens(weekly_block, context_budget, tokenizer_kind, enc)
        weekly_tokens = _count_tokens(weekly_block, tokenizer_kind, enc)

    remaining_after_weekly = max(0, context_budget - weekly_tokens)
    if retrieved_block:
        retrieved_tokens = _count_tokens(retrieved_block, tokenizer_kind, enc)
        if retrieved_tokens > remaining_after_weekly:
            retrieved_block = _truncate_to_tokens(retrieved_block, remaining_after_weekly, tokenizer_kind, enc)

    final_parts = [
        "BUSINESS_CONTEXT:",
        business_context,
        "",
        "WEEKLY_CONTEXT:",
        weekly_block.strip() if weekly_block else "{}",
        "",
        "YDAY_KPI_JSON:",
        kpi_text,
        "",
        "RETRIEVED_EVIDENCE:",
        retrieved_block.strip() if retrieved_block else "[none]",
        "",
        "INSTRUCTIONS_AND_SCHEMA:",
        schema_and_instructions,
    ]
    full_prompt = "\n".join([p for p in final_parts if p is not None])

    # Final safety: drop retrieved if still over budget
    total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)
    if total_tokens > input_budget and retrieved_block:
        retrieved_block = ""
        full_prompt = full_prompt.replace(
            "\nRETRIEVED_EVIDENCE:\n" + retrieved_block.strip() + "\n",
            "\nRETRIEVED_EVIDENCE:\n[none]\n",
        )
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if debug:
        print("DEBUG: final_prompt_tokens =", _count_tokens(full_prompt, tokenizer_kind, enc))
        print("DEBUG: input_budget_tokens =", input_budget)

    # Call the API and return only the raw output string
    resp = client.responses.create(
        model=model,
        input=full_prompt,
        max_output_tokens=max_output_tokens,
    )

    raw = getattr(resp, "output_text", None)
    if not raw:
        out = getattr(resp, "output", None)
        if isinstance(out, list):
            try:
                pieces: List[str] = []
                for item in out:
                    content = getattr(item, "content", None)
                    if isinstance(content, list):
                        for part in content:
                            ptype = getattr(part, "type", None) if not isinstance(part, dict) else part.get("type")
                            if ptype in ("output_text", "text"):
                                ptext = getattr(part, "text", None) if not isinstance(part, dict) else part.get("text")
                                if ptext:
                                    pieces.append(str(ptext))
                raw = "\n".join(pieces).strip() if pieces else None
            except Exception:
                raw = None
        if not raw:
            raw = repr(out) if out is not None else repr(resp)

    return {"raw_model_output": str(raw)}


# Exporting Report

### Concatenating Text

In [0]:
from __future__ import annotations

import ast
import json
import re
from typing import Any, Dict, List, Optional, Tuple


# -------------------------
# Robust JSON extraction + salvage
# -------------------------
def _clean_text_for_json(raw: str) -> str:
    """Normalize common LLM artifacts without being overly destructive."""
    if not isinstance(raw, str):
        raw = str(raw)

    # Strip markdown fences if present
    raw = re.sub(r"^\s*```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s*```\s*$", "", raw)

    # Normalize smart quotes
    raw = (
        raw.replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )

    # Remove ASCII control chars except whitespace
    raw = "".join(ch for ch in raw if (ord(ch) >= 32) or ch in "\n\r\t")
    return raw


def _basic_repairs(s: str) -> str:
    """Small repairs that often fix 'almost JSON' outputs."""
    # Remove trailing commas before closing braces/brackets
    s = re.sub(r",\s*(\}|])", r"\1", s)

    # Convert Python literals to JSON literals
    s = re.sub(r"\bNone\b", "null", s)
    s = re.sub(r"\bTrue\b", "true", s)
    s = re.sub(r"\bFalse\b", "false", s)
    return s


def extract_and_parse_json(raw: str) -> Tuple[Any, str]:
    """
    Parse JSON from a string that may contain extra text before/after the JSON.

    Returns: (parsed_obj, cleaned_json_substring_used)
    Raises: ValueError if nothing JSON-like can be parsed.
    """
    s = _basic_repairs(_clean_text_for_json(raw)).strip()

    # 1) Direct parse (fast path)
    try:
        return json.loads(s), s
    except json.JSONDecodeError:
        pass

    # 2) Find first valid JSON object/array anywhere in the string (robust path)
    dec = json.JSONDecoder()
    starts = [m.start() for m in re.finditer(r"[\{\[]", s)]
    for i in starts:
        try:
            obj, end = dec.raw_decode(s[i:])
            json_sub = s[i : i + end]
            return obj, json_sub
        except json.JSONDecodeError:
            continue

    # 3) Fallback: handle python-ish dict strings (single quotes, etc.)
    try:
        py_obj = ast.literal_eval(s)

        # If it's something like {'raw_model_output': '{...json...}'}, try inner
        if isinstance(py_obj, dict):
            for k in ("raw_model_output", "raw", "output", "raw_output"):
                inner = py_obj.get(k)
                if isinstance(inner, str):
                    return extract_and_parse_json(inner)

        return py_obj, s
    except Exception:
        pass

    raise ValueError("No valid JSON object/array found in input.")


# -------------------------
# Formatting helpers (drivers + alerts)
# -------------------------
def _format_driver(driver: Dict[str, Any], index: int) -> str:
    title = driver.get("title", "Untitled driver")
    what = driver.get("what_happened", driver.get("what", "No description provided."))
    what_it_drove = driver.get("what_it_drove", "Downstream impact not specified.")
    next_steps = driver.get("next_steps", []) or []
    evidence = driver.get("evidence", "none")

    lines: List[str] = []
    lines.append(f"{index}. {title}")
    lines.append(f"   What happened: {what}")
    lines.append(f"   Impact: {what_it_drove}")
    if next_steps:
        lines.append("   Next steps:")
        for i, step in enumerate(next_steps[:5], start=1):
            lines.append(f"     {i}) {step}")
    else:
        lines.append("   Next steps: (none provided)")
    lines.append(f"   Evidence: {evidence}")
    return "\n".join(lines)


def _format_alert(alert: Dict[str, Any], index: int) -> str:
    aid = alert.get("id", f"ALERT_{index}")
    title = alert.get("title") or alert.get("metric") or alert.get("reason") or "Alert"
    severity = alert.get("severity") or alert.get("priority") or "unknown"
    desc = alert.get("description") or alert.get("reason") or ""
    metric = alert.get("metric")
    current = alert.get("current")
    baseline = alert.get("baseline")
    delta_abs = alert.get("delta_abs")
    delta_pct = alert.get("delta_pct")
    recommended = alert.get("recommended_action") or alert.get("recommended_action_text") or ""
    evidence = alert.get("evidence", "none")

    lines: List[str] = []
    lines.append(f"{index}. [{aid}] {title} (severity: {severity})")
    if metric is not None:
        lines.append(f"   Metric: {metric}")
    if current is not None or baseline is not None:
        lines.append(f"   Current: {current} | Baseline: {baseline}")
    if delta_abs is not None or delta_pct is not None:
        dp = f"{delta_pct:.4f}" if isinstance(delta_pct, float) else str(delta_pct)
        lines.append(f"   Delta: abs={delta_abs} pct={dp}")
    if desc:
        lines.append(f"   Why: {desc}")
    if recommended:
        lines.append(f"   Recommended action: {recommended}")
    lines.append(f"   Evidence: {evidence}")
    return "\n".join(lines)


# -------------------------
# Final function: accept summary, drivers, alarms_result and return concatenated text
# -------------------------
def finalize_report(
    *,
    summary: str,
    drivers_payload: Dict[str, Any],
    alarms_result: Any,
    max_driver_items: Optional[int] = None,
    max_alert_items: Optional[int] = None,
) -> Dict[str, Any]:
    """
    summary: executive summary text (string)
    drivers_payload: dict returned by generate_step_level_drivers (expects key 'drivers' -> list)
    alarms_result: either the raw string (from generate_out_of_scope_analyses_and_alarms
                   return value, e.g. result["raw_model_output"]) OR the entire dict returned
                   by that function (which may be {"raw_model_output": "..."}).

    Returns a dict:
      {
        "combined_text": "<summary + drivers + alerts human text>",
        "summary_text": summary,
        "drivers_text": "<formatted drivers>",
        "alerts_text": "<formatted alerts or raw fallback>",
        "parsed_alerts": <parsed alerts list or dict or None>,
        "cleaned_alerts_raw": "<cleaned JSON used to parse or original raw>"
      }
    """
    # 1) Prepare drivers_text
    drivers_list = drivers_payload.get("drivers") if isinstance(drivers_payload.get("drivers"), list) else []
    if max_driver_items is not None:
        drivers_list = drivers_list[:max_driver_items]

    driver_blocks: List[str] = []
    for i, d in enumerate(drivers_list, start=1):
        driver_blocks.append(_format_driver(d, i))
    drivers_text = "\n\n".join(driver_blocks) if driver_blocks else "No drivers provided."

    # 2) Extract raw alerts string from alarms_result in flexible ways
    if isinstance(alarms_result, dict):
        alerts_raw_candidate: Optional[str] = None
        for key in ("raw_model_output", "raw", "output", "raw_output"):
            if key in alarms_result and isinstance(alarms_result[key], str):
                alerts_raw_candidate = alarms_result[key]
                break
        # fallback to repr
        alerts_raw_candidate = alerts_raw_candidate or repr(alarms_result)
    elif isinstance(alarms_result, str):
        alerts_raw_candidate = alarms_result
    else:
        alerts_raw_candidate = repr(alarms_result)

    # 3) Parse alerts JSON robustly (no brittle slicing)
    parsed_alerts = None
    cleaned_alerts_raw = alerts_raw_candidate
    alerts_text = ""

    try:
        parsed_obj, cleaned = extract_and_parse_json(alerts_raw_candidate)
        cleaned_alerts_raw = cleaned

        # Normalize alerts list extraction
        if isinstance(parsed_obj, dict) and isinstance(parsed_obj.get("alerts"), list):
            alerts_list = parsed_obj["alerts"]
        elif isinstance(parsed_obj, list):
            alerts_list = parsed_obj
        elif isinstance(parsed_obj, dict):
            alerts_list = [parsed_obj]
        else:
            alerts_list = []

        parsed_alerts = alerts_list

        if max_alert_items is not None:
            alerts_list = alerts_list[:max_alert_items]

        alert_blocks = []
        for i, a in enumerate(alerts_list, start=1):
            # defensive: only format dict-like alerts
            if isinstance(a, dict):
                alert_blocks.append(_format_alert(a, i))
            else:
                alert_blocks.append(f"{i}. Unstructured alert item: {a!r}")

        alerts_text = "\n\n".join(alert_blocks) if alert_blocks else "No alerts found in parsed output."

    except Exception:
        # parsing failed: preserve raw candidate string for manual parsing
        alerts_text = f"Could not parse alerts JSON. Raw output used for parsing:\n\n{alerts_raw_candidate}"
        parsed_alerts = None
        cleaned_alerts_raw = alerts_raw_candidate

    # 4) Combine summary, drivers, alerts into a single human-readable text
    combined_parts: List[str] = []
    if summary:
        combined_parts.append("=== EXECUTIVE SUMMARY ===")
        combined_parts.append(summary.strip())
    combined_parts.append("\n=== STEP-LEVEL DRIVERS ===")
    combined_parts.append(drivers_text)
    combined_parts.append("\n=== ALERTS / OUT-OF-SCOPE ANALYSES ===")
    combined_parts.append(alerts_text)
    combined_text = "\n\n".join(combined_parts)

    return {
        "combined_text": combined_text,
        "summary_text": summary,
        "drivers_text": drivers_text,
        "alerts_text": alerts_text,
        "parsed_alerts": parsed_alerts,
        "cleaned_alerts_raw": cleaned_alerts_raw,
    }





### Send to Slack Channel

#### Sending Straight Text to Slack

In [0]:
# from __future__ import annotations

# import json
# import os
# from typing import Any, Dict, Optional
# from urllib.request import Request, urlopen
# from urllib.error import HTTPError, URLError


# def _read_env_file(path: str) -> Dict[str, str]:
#     """
#     Minimal .env parser:
#       - supports KEY=VALUE
#       - ignores blank lines and comments starting with '#'
#       - strips surrounding single/double quotes on values
#     """
#     env: Dict[str, str] = {}
#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line or line.startswith("#"):
#                 continue
#             if "=" not in line:
#                 continue
#             k, v = line.split("=", 1)
#             k = k.strip()
#             v = v.strip().strip('"').strip("'")
#             if k:
#                 env[k] = v
#     return env


# def post_finalize_report_to_slack(
#     *,
#     out: Dict[str, Any],
#     env_file: str = ".env",
#     webhook_env_key: str = "SLACK_WEBHOOK_URL",
#     text_key: str = "combined_text",
#     username: Optional[str] = None,
#     icon_emoji: Optional[str] = None,
#     timeout_s: int = 20,
# ) -> None:
#     """
#     Sends the finalize_report() output `out` to Slack via an Incoming Webhook.

#     - Pulls webhook URL from:
#         1) OS environment variable `webhook_env_key`, else
#         2) `env_file` (default ".env") under the same key.
#     - Sends `out[text_key]` (default "combined_text") as the Slack message text.
#     """

#     if not isinstance(out, dict):
#         raise TypeError("out must be a dict (the finalize_report() return value).")

#     # 1) Resolve webhook URL from env var first, then .env file
#     webhook_url = os.getenv(webhook_env_key)
#     if not webhook_url:
#         if not os.path.exists(env_file):
#             raise RuntimeError(
#                 f"Missing Slack webhook URL: {webhook_env_key} not set and env file not found at {env_file!r}."
#             )
#         env_vars = _read_env_file(env_file)
#         webhook_url = env_vars.get(webhook_env_key)

#     if not webhook_url:
#         raise RuntimeError(
#             f"Missing Slack webhook URL: set {webhook_env_key} in your environment or in {env_file!r}."
#         )

#     # 2) Choose message text from out
#     if text_key not in out:
#         available = ", ".join(sorted(out.keys()))
#         raise KeyError(f"out does not contain {text_key!r}. Available keys: {available}")

#     text_val = out[text_key]
#     if text_val is None:
#         raise ValueError(f"out[{text_key!r}] is None; expected a string.")
#     text = str(text_val)

#     # 3) Build Slack payload (plain-text message)
#     payload: Dict[str, Any] = {"text": text}
#     if username:
#         payload["username"] = username
#     if icon_emoji:
#         payload["icon_emoji"] = icon_emoji

#     data = json.dumps(payload).encode("utf-8")
#     req = Request(
#         webhook_url,
#         data=data,
#         headers={"Content-Type": "application/json; charset=utf-8"},
#         method="POST",
#     )

#     # 4) POST to Slack webhook
#     try:
#         with urlopen(req, timeout=timeout_s) as resp:
#             body = resp.read().decode("utf-8", errors="replace")
#             if resp.status >= 300:
#                 raise RuntimeError(f"Slack webhook failed: HTTP {resp.status} body={body}")
#     except HTTPError as e:
#         raise RuntimeError(
#             f"Slack webhook HTTPError: {e.code} {e.read().decode('utf-8', 'replace')}"
#         ) from e
#     except URLError as e:
#         raise RuntimeError(f"Slack webhook URLError: {e}") from e



#### Saving Doc File & Sharing Link

In [0]:
from __future__ import annotations

import json
import os
import time
import uuid
from typing import Any, Dict, Optional
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError


def _read_env_file(path: str) -> Dict[str, str]:
    """
    Minimal .env parser:
      - supports KEY=VALUE
      - ignores blank lines and comments starting with '#'
      - strips surrounding single/double quotes on values
    """
    env: Dict[str, str] = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = k.strip()
            v = v.strip().strip('"').strip("'")
            if k:
                env[k] = v
    return env

def post_finalize_report_to_slack(
    *,
    out: Dict[str, Any],
    env_file: str = ".env",
    webhook_env_key: str = "SLACK_WEBHOOK_URL",
    text_key: str = "combined_text",
    username: Optional[str] = None,
    icon_emoji: Optional[str] = None,
    timeout_s: int = 20,
) -> None:
    """
    Sends the finalize_report() output `out` to Slack.

    Behavior (Option A):
    - Try to find a local/driver-readable .docx path in `out`.
    - If found and SLACK_BOT_TOKEN is available, upload the file via Slack Web API (files.upload).
      * If SLACK_UPLOAD_CHANNEL is set (env or env_file), upload will post into that channel.
      * Otherwise upload without channels and use returned permalink to include a link in the message.
    - If no bot token / upload fails / no file found: fallback to original webhook-only behavior (post text).
    """
    if not isinstance(out, dict):
        raise TypeError("out must be a dict (the finalize_report() return value).")

    # Resolve env file variables helper
    def _resolve_from_env_or_file(key: str) -> Optional[str]:
        v = os.getenv(key)
        if v:
            return v
        if os.path.exists(env_file):
            try:
                env_vars = _read_env_file(env_file)
                return env_vars.get(key)
            except Exception:
                return None
        return None

    # 1) Resolve webhook URL for fallback posting (keep original behavior)
    webhook_url = _resolve_from_env_or_file(webhook_env_key)
    # do not raise yet — we may use bot token path instead; raise only if we need webhook and it's missing

    # 2) Validate text key
    if text_key not in out:
        available = ", ".join(sorted(out.keys()))
        raise KeyError(f"out does not contain {text_key!r}. Available keys: {available}")
    text_val = out[text_key]
    if text_val is None:
        raise ValueError(f"out[{text_key!r}] is None; expected a string.")
    text = str(text_val)

    # 3) Find docx path in out (same candidate keys as before)
    candidate_docx_keys = (
        "docx_path",
        "doc_path",
        "document_path",
        "report_docx_path",
        "output_docx_path",
        "file_path",
        "path",
    )

    docx_path: Optional[str] = None
    for k in candidate_docx_keys:
        v = out.get(k)
        if isinstance(v, str) and v.lower().endswith(".docx") and v.strip():
            docx_path = v.strip()
            break

    if not docx_path:
        artifacts = out.get("artifacts")
        if isinstance(artifacts, dict):
            for k in candidate_docx_keys:
                v = artifacts.get(k)
                if isinstance(v, str) and v.lower().endswith(".docx") and v.strip():
                    docx_path = v.strip()
                    break

    # Also allow a more explicit key that may be present
    if not docx_path:
        alt = out.get("report_docx_path") or out.get("report_path")
        if isinstance(alt, str) and alt.lower().endswith(".docx"):
            docx_path = alt

    # 4) Resolve driver-local path candidates
    driver_local_path: Optional[str] = None
    if docx_path:
        # direct local or dbfs style resolution attempts
        attempts: List[str] = []
        # If starts with dbfs:/ convert to /dbfs/...
        if docx_path.startswith("dbfs:/"):
            attempts.append("/dbfs/" + docx_path[len("dbfs:/") :])
        # If starts with /dbfs/ use it
        if docx_path.startswith("/dbfs/"):
            attempts.append(docx_path)
        # As-is (maybe already a local path)
        attempts.append(docx_path)
        # As a basename under /tmp (common pattern)
        basename = os.path.basename(docx_path)
        if basename:
            attempts.append(os.path.join("/tmp", basename))
            # also check CWD (just in case)
            attempts.append(os.path.join(os.getcwd(), basename))

        for p in attempts:
            try:
                if p and os.path.exists(p) and os.path.isfile(p):
                    driver_local_path = p
                    break
            except Exception:
                # ignore permission issues; we'll fallback to webhook
                continue

    # 5) Try to upload to Slack via files.upload (bot token)
    slack_bot_token = _resolve_from_env_or_file("SLACK_BOT_TOKEN")
    slack_upload_channel = _resolve_from_env_or_file("SLACK_UPLOAD_CHANNEL") or _resolve_from_env_or_file("SLACK_CHANNEL")

    file_permalink: Optional[str] = None
    uploaded_filename: Optional[str] = None
    upload_succeeded = False

    if driver_local_path and slack_bot_token:
        # try to upload using requests (multipart). If requests missing or upload fails, fallback.
        try:
            import requests  # type: ignore
        except Exception:
            requests = None  # type: ignore

        if requests is not None:
            try:
                headers = {"Authorization": f"Bearer {slack_bot_token}"}
                data = {}
                if slack_upload_channel:
                    data["channels"] = slack_upload_channel
                # Use an initial comment referencing the text (short)
                short_comment = (text[:1000] + "...") if len(text) > 1000 else text
                data["initial_comment"] = short_comment
                # Open file and post
                with open(driver_local_path, "rb") as fh:
                    files = {"file": (os.path.basename(driver_local_path), fh, "application/vnd.openxmlformats-officedocument.wordprocessingml.document")}
                    resp = requests.post("https://slack.com/api/files.upload", headers=headers, data=data, files=files, timeout=timeout_s)
                try:
                    j = resp.json()
                except Exception:
                    raise RuntimeError(f"Slack upload returned non-JSON (HTTP {resp.status_code}): {resp.text[:500]!r}")
                if not j.get("ok"):
                    # If error is 'not_authed' or 'invalid_auth', we want to fallback to webhook
                    err = j.get("error", "<no error field>")
                    raise RuntimeError(f"Slack files.upload error: {err} (HTTP {resp.status_code})")
                # success
                file_obj = j.get("file", {})
                file_permalink = file_obj.get("permalink") or file_obj.get("permalink_public")
                uploaded_filename = file_obj.get("name") or os.path.basename(driver_local_path)
                upload_succeeded = True
            except Exception as e:
                # swallow and fallback to webhook path below
                upload_succeeded = False
                upload_error_msg = str(e)
        else:
            upload_succeeded = False
            upload_error_msg = "requests library not available"
    else:
        # no local path or no bot token; can't use files.upload
        upload_succeeded = False

    # 6) Build payload (if upload succeeded include link/button; else fallback to text via webhook)
    if upload_succeeded and file_permalink:
        # Use blocks with a button linking to the Slack file permalink
        payload: Dict[str, Any] = {
            "text": f"{text}\n\n📄 {uploaded_filename}: {file_permalink}",
            "blocks": [
                {"type": "section", "text": {"type": "mrkdwn", "text": text}},
                {
                    "type": "actions",
                    "elements": [
                        {
                            "type": "button",
                            "text": {"type": "plain_text", "text": "Open file in Slack"},
                            "url": file_permalink,
                        }
                    ],
                },
            ],
        }
        # add small context element showing uploaded filename
        if uploaded_filename:
            payload["blocks"].insert(1, {"type": "context", "elements": [{"type": "mrkdwn", "text": f"File: `{uploaded_filename}` uploaded to Slack"}]})
    elif upload_succeeded and not file_permalink:
        # uploaded but no permalink: still send a helpful message
        payload = {"text": f"{text}\n\n📄 File uploaded to Slack (no permalink available)."}
    else:
        # fallback: use the incoming webhook to post text (original behavior)
        if not webhook_url:
            # nothing we can do: no webhook and no bot token upload
            raise RuntimeError(
                "Unable to upload file to Slack (no SLACK_BOT_TOKEN or upload failed) and no webhook URL available to post text."
            )
        payload: Dict[str, Any] = {"text": text}

    # preserve username/icon if provided (works for incoming webhook; ignored for files.upload initial_comment which we already set)
    if username:
        payload["username"] = username
    if icon_emoji:
        payload["icon_emoji"] = icon_emoji

    # 7) If we uploaded via files.upload and provided channel, we're already posted to that channel.
    #    However, we'll still call webhook to provide a nicely formatted message with a button that links to the uploaded file's permalink (if available).
    if upload_succeeded and not webhook_url:
        # We uploaded and there's no webhook: nothing more to do (file already posted if channels was provided).
        return None

    # 8) POST the payload to webhook if webhook is available (either as primary or fallback)
    if webhook_url:
        data = json.dumps(payload).encode("utf-8")
        req = Request(
            webhook_url,
            data=data,
            headers={"Content-Type": "application/json; charset=utf-8"},
            method="POST",
        )
        try:
            with urlopen(req, timeout=timeout_s) as resp:
                body = resp.read().decode("utf-8", errors="replace")
                if resp.status >= 300:
                    raise RuntimeError(f"Slack webhook failed: HTTP {resp.status} body={body}")
        except HTTPError as e:
            raise RuntimeError(f"Slack webhook HTTPError: {e.code} {e.read().decode('utf-8', 'replace')}") from e
        except URLError as e:
            raise RuntimeError(f"Slack webhook URLError: {e}") from e

    return None


# Cleanup

### Add Report to Knowledge

In [0]:
def save_final_report_to_knowledge_docx(
    *,
    final_report: Dict[str, Any],
    knowledge_dir: str | Path = "knowledge",
    filename: Optional[str] = None,  # e.g. "01_30_2026_report.docx"
    overwrite: bool = False,
) -> str:
    """
    Save finalize_report() output as a DOCX in <repo_root>/knowledge/previousReports
    (not relative to the current working directory).

    Uses final_report["combined_text"] as the document body.

    Returns the path to the DOCX written.
    """
    if not isinstance(final_report, dict):
        raise TypeError("final_report must be a dict (finalize_report() return value).")

    combined_text = final_report.get("combined_text")
    if not combined_text or not str(combined_text).strip():
        raise ValueError("final_report is missing non-empty 'combined_text'.")

    # ---- find repo root by walking upward until we see a "knowledge" dir or a marker file ----
    # Adjust markers if your repo uses something else (pyproject.toml, setup.cfg, etc.)
    start = Path.cwd().resolve()
    repo_root: Optional[Path] = None
    for p in [start] + list(start.parents):
        # prefer an actual "knowledge" folder if present
        if (p / "knowledge").exists():
            repo_root = p
            break
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            repo_root = p
            break

    # Fallback: if nothing found, default to cwd (won't crash, but may create shallow dirs)
    repo_root = repo_root or start

    # If we accidentally resolved to a "src" directory (common when running from inside src),
    # move up one level to the repository root so knowledge/ ends up at the true repo root.
    # Also, if the chosen repo_root doesn't contain "knowledge" but its parent does, prefer the parent.
    if repo_root.name == "src":
        repo_root = repo_root.parent
    elif not (repo_root / "knowledge").exists() and (repo_root.parent / "knowledge").exists():
        repo_root = repo_root.parent

    # Always write to <repo_root>/knowledge/previousReports
    reports_dir = (Path(repo_root) / Path(knowledge_dir) / "previousReports").resolve()
    reports_dir.mkdir(parents=True, exist_ok=True)

    if filename is None:
        date_key = time.strftime("%m_%d_%Y")
        filename = f"{date_key}_report.docx"

    if not filename.lower().endswith(".docx"):
        filename += ".docx"

    out_path = reports_dir / filename

    if out_path.exists() and not overwrite:
        raise FileExistsError(
            f"Report already exists: {out_path}. Pass overwrite=True to replace."
        )

    doc = Document()
    doc.add_heading("Daily KPI Report", level=1)

    for line in str(combined_text).splitlines():
        doc.add_paragraph("" if line.strip() == "" else line)

    doc.save(str(out_path))
    return str(out_path)


### Summarize Report & Save

In [0]:
def summarize_current_report_to_summaries(
    *,
    report_docx_path: str | Path,
    knowledge_dir: str | Path = "knowledge",
    model: str = "gpt-5-mini",
    max_report_chars: int = 60_000,
    overwrite: bool = False,
    output_filename: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Summarize ONE current report DOCX and save the summary as a DOCX in
    <repo_root>/knowledge/summaries (not relative to current working directory).

    Self-contained: does not require _make_prompt or _read_docx_text.
    """
    report_docx_path = Path(report_docx_path).resolve()
    if not report_docx_path.exists():
        raise FileNotFoundError(f"Report not found: {report_docx_path}")

    # ---- find repo root by walking upward from report path ----
    start_dir = report_docx_path.parent
    repo_root: Optional[Path] = None
    for p in [start_dir] + list(start_dir.parents):
        if (p / "knowledge").exists():
            repo_root = p
            break
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            repo_root = p
            break
    repo_root = repo_root or Path.cwd().resolve()

    # If we accidentally resolved to a "src" directory (common when running from inside src),
    # move up one level to the repository root so knowledge/ ends up at the true repo root.
    # Also prefer the parent if it contains "knowledge" even when repo_root doesn't.
    if repo_root.name == "src":
        repo_root = repo_root.parent
    elif not (repo_root / "knowledge").exists() and (repo_root.parent / "knowledge").exists():
        repo_root = repo_root.parent

    summaries_dir = (Path(repo_root) / Path(knowledge_dir) / "summaries").resolve()
    summaries_dir.mkdir(parents=True, exist_ok=True)

    # Parse date from filename or fallback to yesterday
    m = _REPORT_DATE_RE.match(report_docx_path.name)
    dt = datetime(int(m["yyyy"]), int(m["mm"]), int(m["dd"])) if m else datetime.now()
    date_key = dt.strftime("%m_%d_%Y")
    day_of_week = dt.strftime("%A")

    # Read report text from DOCX
    doc_in = Document(str(report_docx_path))
    parts: List[str] = []
    for p in doc_in.paragraphs:
        t = (p.text or "").strip()
        if t:
            parts.append(t)
    report_text = "\n".join(parts)[:max_report_chars]

    # Build prompt locally
    prompt = f"""
You are summarizing a daily operations KPI report.

Report date: {date_key}

TASK
A) EXECUTIVE SUMMARY
- Write a 5–6 sentence executive summary capturing overall performance direction (better/worse/mixed),
  the biggest drivers, and any notable anomalies or consistency with prior days.
- Do NOT include specific numbers.

B) CALLOUTS
- Provide 3–7 smaller trend callouts (step-level rates, queue/breakage, bot vs non-bot, funnel shifts, etc.).
- Each callout must be ONE short sentence, no numbers.
- Each callout must include an importance label: High | Medium | Low.

OUTPUT FORMAT (STRICT JSON ONLY)
{{
  "executive_summary": "<5-6 sentences>",
  "callouts": [
    {{"importance": "High", "text": "<callout sentence>"}},
    ...
  ]
}}

REPORT TEXT
\"\"\"
{report_text}
\"\"\"
""".strip()

    client = OpenAI()
    resp = client.responses.create(model=model, input=prompt)
    raw = (resp.output_text or "").strip()

    # Parse JSON (fallback: extract first JSON object)
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        start_i = raw.find("{")
        end_i = raw.rfind("}")
        if start_i == -1 or end_i == -1 or end_i <= start_i:
            raise ValueError(f"Model did not return JSON. Output was:\n{raw}")
        data = json.loads(raw[start_i : end_i + 1])

    exec_summary = str(data.get("executive_summary", "")).strip()
    callouts = data.get("callouts") if isinstance(data.get("callouts"), list) else []

    # Normalize callouts
    cleaned_callouts: List[Dict[str, str]] = []
    for c in callouts:
        if not isinstance(c, dict):
            continue
        importance = str(c.get("importance", "")).strip().title()
        if importance not in {"High", "Medium", "Low"}:
            importance = "Medium"
        text = str(c.get("text", "")).strip()
        if text:
            cleaned_callouts.append({"importance": importance, "text": text})

    # Output filename
    if output_filename is None:
        output_filename = f"{date_key}_summary.docx"
    if not output_filename.lower().endswith(".docx"):
        output_filename += ".docx"

    summary_docx_path = (summaries_dir / output_filename).resolve()
    if summary_docx_path.exists() and not overwrite:
        raise FileExistsError(
            f"Summary already exists: {summary_docx_path}. Pass overwrite=True to replace."
        )

    # Write summary docx
    doc_out = Document()
    doc_out.add_heading(f"Summary - {date_key} ({day_of_week})", level=1)
    doc_out.add_paragraph(exec_summary if exec_summary else "(No executive summary returned.)")
    doc_out.add_heading("Callouts (with importance)", level=2)

    if cleaned_callouts:
        for c in cleaned_callouts:
            doc_out.add_paragraph(f"[{c['importance']}] {c['text']}", style="List Bullet")
    else:
        doc_out.add_paragraph("(No callouts returned.)")

    doc_out.save(str(summary_docx_path))

    return {
        "summary_docx_path": str(summary_docx_path),
        "executive_summary": exec_summary,
        "callouts": cleaned_callouts,
        "raw_model_output": raw,
    }


# Main Execution

In [0]:
from __future__ import annotations

import os
import time
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple

from dotenv import load_dotenv
from tqdm import tqdm

def _stamp(msg: str, t0: Optional[float] = None) -> None:
    """
    Lightweight logger for pipeline steps.

    Args:
        msg: Message to print.
        t0: Optional start time (from time.time()) to print elapsed seconds.
    """
    # Toggle with LOG_STAMPS=0/false/no to silence
    val = os.getenv("LOG_STAMPS", "1").strip().lower()
    if val in {"0", "false", "no", "off"}:
        return

    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    if t0 is None:
        print(f"[{ts}] {msg}")
    else:
        elapsed = time.time() - float(t0)
        print(f"[{ts}] {msg} (+{elapsed:0.2f}s)")


# ============================================================
# 1) Start -> JSON creation
# ============================================================
def step1_build_kpi_json(
    *,
    company_id: int = 25,
    call_direction: str = "INBOUND",
    restrict_to_yday_and_prior4: bool = True,
    allowed_buckets: Optional[Set[str]] = None,
    site_serp_values: Sequence[str] = ("Site", "SERP"),
    # compute_kpis params
    create_temp_view_for_kpis: bool = True,
    debug_print_sql: bool = False,
    # filter params
    min_metric_current_calls: int = 5,
    min_abs_delta_pct: float = 0.01,
    min_breakdown_current_calls: int = 5,
    breakdown_top_n: int = 5,
) -> Dict[str, Any]:
    """
    Builds KPI tables (overall + buckets + site/serp) -> kpi_json_full -> kpi_json_filtered.
    Returns everything needed for downstream steps.
    """
    if allowed_buckets is None:
        allowed_buckets = {
            "Natural",
            "Brand-Partner",
            "Generic",
            "Aggregator",
            "Competitor",
            "Utility",
            "PMax",
            "NRG",
            "Other Bucket",
        }

    t0 = time.time()
    _stamp("STEP 1 START: KPI build -> JSON", t0)

    section_names: List[str] = []
    section_tables: List[Any] = []

    # OVERALL
    _stamp("STEP 1A: overall", t0)
    calls_df_overall = get_data(
        company_id=company_id,
        call_direction=call_direction,
        restrict_to_yday_and_prior4=restrict_to_yday_and_prior4,
        create_temp_view=False,
        marketing_buckets=None,
        site_serp=None,
    )
    kpi_df_overall = compute_kpis(
        calls_df=calls_df_overall,
        source_view="calls_full_debug_overall",
        create_temp_view=create_temp_view_for_kpis,
        debug_print_sql=debug_print_sql,
    )
    section_names.append("overall")
    section_tables.append(kpi_df_overall)

    # BUCKETS
    buckets_sorted = sorted(list(allowed_buckets))
    _stamp(f"STEP 1B: buckets | n={len(buckets_sorted)}", t0)
    for bucket in tqdm(buckets_sorted, desc="Marketing buckets"):
        bucket_snake = _to_snake(bucket)

        calls_df_bucket = get_data(
            company_id=company_id,
            call_direction=call_direction,
            restrict_to_yday_and_prior4=restrict_to_yday_and_prior4,
            create_temp_view=False,
            marketing_buckets=bucket,
            site_serp=None,
        )
        kpi_df_bucket = compute_kpis(
            calls_df=calls_df_bucket,
            source_view=f"calls_full_debug_bucket_{bucket_snake}",
            create_temp_view=create_temp_view_for_kpis,
            debug_print_sql=debug_print_sql,
        )

        section_names.append(f"bucket_{bucket_snake}")
        section_tables.append(kpi_df_bucket)

    # SITE/SERP
    _stamp(f"STEP 1C: site/serp | values={list(site_serp_values)}", t0)
    for ss in tqdm(site_serp_values, desc="Site vs SERP"):
        ss_snake = _to_snake(ss)

        calls_df_ss = get_data(
            company_id=company_id,
            call_direction=call_direction,
            restrict_to_yday_and_prior4=restrict_to_yday_and_prior4,
            create_temp_view=False,
            marketing_buckets=None,
            site_serp=ss,
        )
        kpi_df_ss = compute_kpis(
            calls_df=calls_df_ss,
            source_view=f"calls_full_debug_{ss_snake}",
            create_temp_view=create_temp_view_for_kpis,
            debug_print_sql=debug_print_sql,
        )

        section_names.append(ss_snake)
        section_tables.append(kpi_df_ss)

    # JSON build + filter
    _stamp(f"STEP 1D: JSON build | sections={len(section_names)}", t0)
    kpi_json_full = kpi_tables_to_json(names=section_names, tables=section_tables)

    _stamp("STEP 1E: filter_and_compact_kpi_json()", t0)
    kpi_json_filtered = filter_and_compact_kpi_json(
        kpi_json_full,
        min_metric_current_calls=min_metric_current_calls,
        min_abs_delta_pct=min_abs_delta_pct,
        min_breakdown_current_calls=min_breakdown_current_calls,
        breakdown_top_n=breakdown_top_n,
    )

    _stamp("STEP 1 DONE", t0)
    return {
        "kpi_json_full": kpi_json_full,
        "kpi_json_filtered": kpi_json_filtered,
        "section_names": section_names,
        "section_tables": section_tables,
    }
# robust step2_build_context: searches upward for .env and loads it
def _find_upward_env(start_path: Optional[str] = None, filename: str = ".env", max_levels: int = 10) -> Optional[str]:
    """
    Search upward from start_path (or cwd) up to max_levels for filename.
    Returns the absolute path to the first match, or None.
    """
    from pathlib import Path

    p = Path(start_path) if start_path else Path.cwd()
    p = p.resolve()
    for _ in range(max_levels + 1):
        candidate = p / filename
        if candidate.exists() and candidate.is_file():
            return str(candidate)
        if p.parent == p:
            break
        p = p.parent
    return None


def step2_build_context(
    *,
    kpi_json_filtered: Any,
    knowledge_dir: str = "knowledge",
    lookback_summaries: int = 3,
    n_queries: int = 6,
    model: str = "gpt-5-mini",
    max_results_per_query: int = 5,
    base_business_context: Optional[str] = None,
    env_file: str = ".env",  # can be relative or absolute
    vector_store_env_key: str = "OPENAI_VECTOR_STORE_ID",
) -> Dict[str, Any]:
    """
    Uses filtered KPI JSON to generate query plan and retrieve context blocks.
    Robustly loads the env file by searching upwards from cwd if needed.
    """
    t0 = time.time()
    _stamp("STEP 2 START: JSON -> context", t0)

    # Attempt 1: if env_file is an absolute or explicit path and exists, use it
    env_path = None
    if env_file:
        p = Path(env_file)
        if p.is_absolute() and p.exists():
            env_path = str(p)
        elif p.exists():
            env_path = str(p.resolve())

    # Attempt 2: search upward from cwd for the filename (only if env_path not already found)
    if not env_path:
        found = _find_upward_env(start_path=None, filename=env_file or ".env", max_levels=20)
        if found:
            env_path = found

    # Debug info: show where we will try to load from
    print(f"[DEBUG] resolved env_path: {env_path!r} (cwd={Path.cwd()})")

    loaded = False
    if env_path:
        # load_dotenv returns True if a file was found and parsed
        try:
            loaded = load_dotenv(dotenv_path=env_path)
            print(f"[DEBUG] load_dotenv('{env_path}') returned: {loaded}")
        except Exception as e:
            print(f"[DEBUG] load_dotenv raised: {e}")

    # Fallback: try plain load_dotenv() (searches common locations)
    if not loaded:
        try:
            loaded = load_dotenv()
            print(f"[DEBUG] load_dotenv() fallback returned: {loaded}")
        except Exception as e:
            print(f"[DEBUG] load_dotenv() fallback raised: {e}")

    # Final fallback: manual parse if file exists but wasn't loaded
    if not loaded and env_path and os.path.exists(env_path):
        print(f"[DEBUG] manual fallback parse for {env_path!r}")
        try:
            with open(env_path, "r", encoding="utf-8") as f:
                for raw in f:
                    line = raw.strip()
                    if not line or line.startswith("#") or "=" not in line:
                        continue
                    k, v = line.split("=", 1)
                    k = k.strip()
                    v = v.strip().strip('"').strip("'")
                    if k:
                        os.environ.setdefault(k, v)
            loaded = True
            print("[DEBUG] manual parse complete")
        except Exception as e:
            print(f"[DEBUG] manual parse failed: {e}")

    vector_store_id = os.getenv(vector_store_env_key)
    print(f"[DEBUG] {vector_store_env_key} present: {bool(vector_store_id)}")

    if not vector_store_id:
        raise ValueError(f"{vector_store_env_key} is not set in the environment/.env (tried: {env_path!r})")

    # --- existing logic continues ---
    weekly_context = get_weekly_context(knowledge_dir, lookback_summaries=lookback_summaries)

    query_plan = generate_vector_store_queries(
        kpi_json=kpi_json_filtered,
        weekly_context=weekly_context,
        n_queries=n_queries,
        model=model,
        base_business_context=base_business_context,
    )

    retrieved_blocks = retrieve_vector_store_blocks(
        vector_store_id=vector_store_id,
        query_plan=query_plan,
        max_results_per_query=max_results_per_query,
        save_path=None,
    )

    context_vars = {
        "kpi_json_filtered": kpi_json_filtered,
        "weekly_context": weekly_context,
        "query_plan": query_plan,
        "retrieved_blocks": retrieved_blocks,
        "vector_store_id": vector_store_id,
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }

    _stamp("STEP 2 DONE", t0)
    return context_vars



# ============================================================
# 3) OpenAI generation calls
# ============================================================
def step3_generate_outputs(
    *,
    kpi_json_filtered: Any,
    context_vars: Dict[str, Any],
    model: str = "gpt-5-mini",
    # alarms params
    min_alerts: int = 3,
    min_monitor_items: int = 4,
    max_output_tokens: int = 2000,
    model_context_window_tokens: int = 16000,
    input_budget_fraction: float = 0.8,
    alarms_debug: bool = True,
) -> Dict[str, Any]:
    """
    Runs the three OpenAI generation functions.
    Returns their raw outputs.
    """
    t0 = time.time()
    _stamp("STEP 3 START: OpenAI generation", t0)

    weekly_context = context_vars.get("weekly_context")
    retrieved_blocks = context_vars.get("retrieved_blocks")

    summary_text = generate_executive_summary_for_report_text(
        kpi_json=kpi_json_filtered,
        weekly_context=weekly_context,
        retrieved_blocks=retrieved_blocks,
        model=model,
    )

    drivers_payload = generate_step_level_drivers(
        kpi_json=kpi_json_filtered,
        weekly_context=None,
        retrieved_blocks=retrieved_blocks,
        min_drivers=3,
    )

    alarms_result = generate_out_of_scope_analyses_and_alarms(
        kpi_json=kpi_json_filtered,
        weekly_context=None,
        retrieved_blocks=retrieved_blocks,
        model=model,
        min_alerts=min_alerts,
        min_monitor_items=min_monitor_items,
        max_output_tokens=max_output_tokens,
        model_context_window_tokens=model_context_window_tokens,
        input_budget_fraction=input_budget_fraction,
        debug=alarms_debug,
    )

    _stamp("STEP 3 DONE", t0)
    return {
        "summary_text": summary_text,
        "drivers_payload": drivers_payload,
        "alarms_result": alarms_result,
    }


# ============================================================
# 4) Format + Save DOCX + Slack
# ============================================================
from datetime import datetime
from typing import Any, Dict, Optional


def step4_finalize_and_send_to_slack(
    *,
    summary_text: str,
    drivers_payload: Dict[str, Any],
    alarms_result: Any,
    # reporting / limits
    max_driver_items: int = 10,
    max_alert_items: int = 10,
    # Slack params
    env_file: str = ".env",
    webhook_env_key: str = "SLACK_WEBHOOK_URL",
    slack_username: Optional[str] = None,
    slack_icon_emoji: Optional[str] = None,
    # Knowledge / DOCX save params (added defaults; safe for existing callers)
    knowledge_dir: str = "knowledge",
    overwrite_report_docx: bool = True,
) -> Dict[str, Any]:
    """
    Finalizes report formatting, saves the report DOCX, and posts to Slack.

    IMPORTANT:
    - We save the DOCX *before* posting to Slack so the Slack function can include a link.
    - The returned `final_report` dict is augmented with `docx_path` and `report_docx_path`
      for downstream use.
    """
    t0 = time.time()
    _stamp("STEP 4 START: finalize + save DOCX + Slack", t0)

    # 1) Build the final report text payload
    final_report = finalize_report(
        summary=summary_text,
        drivers_payload=drivers_payload,
        alarms_result=alarms_result,
        max_driver_items=max_driver_items,
        max_alert_items=max_alert_items,
    )

    # 2) Save report DOCX FIRST (so Slack can link it)
    date_key = datetime.now().strftime("%m_%d_%Y")
    report_docx_path = save_final_report_to_knowledge_docx(
        final_report=final_report,
        knowledge_dir=knowledge_dir,
        filename=f"{date_key}_report.docx",
        overwrite=overwrite_report_docx,
    )
    print("Saved report DOCX:", report_docx_path)

        # 2) Summarize THAT file (don’t hardcode the path)
    summary_out = summarize_current_report_to_summaries(
        report_docx_path=report_docx_path,
        knowledge_dir="knowledge",
        model="gpt-5-mini",
        overwrite=False,
    )

    # Add a key the Slack poster knows how to detect
    final_report["docx_path"] = report_docx_path
    final_report["report_docx_path"] = report_docx_path  # optional extra for clarity

    # 3) Post to Slack (now includes link if your post_finalize_report_to_slack supports it)
    post_finalize_report_to_slack(
        out=final_report,
        env_file=env_file,
        webhook_env_key=webhook_env_key,
        username=slack_username,
        icon_emoji=slack_icon_emoji,
    )

    _stamp("STEP 4 DONE", t0)
    return final_report


# s1=None
# ctx=None
# gen=None
# final_report=None
# from datetime import datetime

# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    # s1 = step1_build_kpi_json()
    # ctx = step2_build_context(kpi_json_filtered=s1["kpi_json_filtered"])
    # gen = step3_generate_outputs(kpi_json_filtered=s1["kpi_json_filtered"], context_vars=ctx)
    final_report = step4_finalize_and_send_to_slack(
        summary_text=gen["summary_text"],
        drivers_payload=gen["drivers_payload"],
        alarms_result=gen["alarms_result"],
    )

    print("Saved summary DOCX:", summary_out["summary_docx_path"])

    print("\n--- Final combined report (truncated) ---\n")
    print(final_report["combined_text"][:4000])


In [0]:
# ctx = step2_build_context(kpi_json_filtered=s1["kpi_json_filtered"])
# gen = step3_generate_outputs(kpi_json_filtered=s1["kpi_json_filtered"], context_vars=ctx)
# final_report = step4_finalize_and_send_to_slack(
#     summary_text=gen["summary_text"],
#     drivers_payload=gen["drivers_payload"],
#     alarms_result=gen["alarms_result"],
# )

# print("\n--- Final combined report (truncated) ---\n")
# print(final_report["combined_text"][:4000])

In [0]:
# final_report = step4_finalize_and_send_to_slack(
#     summary_text=gen["summary_text"],
#     drivers_payload=gen["drivers_payload"],
#     alarms_result=gen["alarms_result"],
# )

In [0]:
# # 1) Save report DOCX (returns the exact filepath)
# date_key = datetime.now().strftime("%m_%d_%Y")
# report_docx_path = save_final_report_to_knowledge_docx(
#     final_report=final_report,
#     knowledge_dir="knowledge",
#     filename=f"{date_key}_report.docx",
#     overwrite=True,
# )
# print("Saved report DOCX:", report_docx_path)

# # 2) Summarize THAT file (don’t hardcode the path)
# summary_out = summarize_current_report_to_summaries(
#     report_docx_path=report_docx_path,
#     knowledge_dir="knowledge",
#     model="gpt-5-mini",
#     overwrite=False,
# )
# print("Saved summary DOCX:", summary_out["summary_docx_path"])

# print("\n--- Final combined report (truncated) ---\n")
# print(final_report["combined_text"][:4000])
